In [1]:

import sys
import torch
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    x = torch.randn(100, 100, device="cuda")
    print("GPU test successful:", x.device)
else:
    print("WARNING: GPU not active")

Python: 3.13.16
PyTorch: 2.11.0+cu130
Pandas: 2.3.3
NumPy: 2.1.3
CUDA available: True
GPU: Tesla T4
GPU test successful: cuda:0


In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT = Path("/kaggle/input")
CUTTERS = [1, 4, 6]

all_csv = list(ROOT.rglob("*.csv"))

print("Total CSV files:", len(all_csv))

for cutter in CUTTERS:
    pattern = re.compile(
        rf"c_{cutter}_(\d{{3}})\.csv",
        re.IGNORECASE
    )

    sensor_files = [
        p for p in all_csv
        if pattern.fullmatch(p.name)
    ]

    wear_files = [
        p for p in all_csv
        if p.name.lower() == f"c{cutter}_wear.csv"
    ]

    print(f"\nCutter C{cutter}")
    print("Sensor files:", len(sensor_files))
    print("Wear files:", len(wear_files))

    assert len(sensor_files) == 315
    assert len(wear_files) == 1

    ids = {
        int(pattern.fullmatch(p.name).group(1))
        for p in sensor_files
    }
    assert ids == set(range(1, 316))

    wear = pd.read_csv(wear_files[0])
    assert len(wear) == 315
    assert set(wear["cut"]) == ids

print("\nDATASET VERIFIED")

# Inspect sample signal length
sample_file = next(
    p for p in all_csv
    if p.name.lower() == "c_1_001.csv"
)

sample = pd.read_csv(
    sample_file,
    header=None,
    dtype=np.float32
)

print("\nExample recording:", sample_file.name)
print("Shape:", sample.shape)
print("Recording duration (seconds):",
      round(len(sample) / 50000, 3))

assert sample.shape[1] == 7
assert np.isfinite(sample.to_numpy()).all()

print("\nReady for signal segmentation.")

Total CSV files: 949

Cutter C1
Sensor files: 315
Wear files: 1

Cutter C4
Sensor files: 315
Wear files: 1

Cutter C6
Sensor files: 315
Wear files: 1

DATASET VERIFIED

Example recording: c_1_001.csv
Shape: (127399, 7)
Recording duration (seconds): 2.548

Ready for signal segmentation.


In [3]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.signal import resample_poly

ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/phm_sequences.npz")

CUTTERS = [1, 4, 6]
WINDOWS_PER_CUT = 4
RAW_WINDOW = 8192
DOWNSAMPLE = 4
SEQ_LEN = RAW_WINDOW // DOWNSAMPLE

all_files = list(ROOT.rglob("*.csv"))

X_all = []
y_all = []
cutter_all = []
cut_all = []

for cutter in CUTTERS:

    wear_paths = [
        p for p in all_files
        if p.name.lower() == f"c{cutter}_wear.csv"
    ]
    assert len(wear_paths) == 1

    wear = pd.read_csv(wear_paths[0])
    wear["wear_max"] = wear[
        ["flute_1", "flute_2", "flute_3"]
    ].max(axis=1)

    assert wear["cut"].is_unique
    assert set(wear["cut"]) == set(range(1, 316))
    assert np.isfinite(wear["wear_max"]).all()

    wear = wear.set_index("cut")

    sensor_paths = {}
    pattern = re.compile(
        rf"c_{cutter}_(\d{{3}})\.csv",
        re.IGNORECASE
    )

    for p in all_files:
        match = pattern.fullmatch(p.name)
        if match:
            cut_id = int(match.group(1))
            assert cut_id not in sensor_paths
            sensor_paths[cut_id] = p

    assert set(sensor_paths) == set(range(1, 316))

    print(f"\nProcessing C{cutter}...", flush=True)

    for cut_id in range(1, 316):

        signal = pd.read_csv(
            sensor_paths[cut_id],
            header=None,
            dtype=np.float32
        ).to_numpy()

        assert signal.shape[1] == 7
        assert np.isfinite(signal).all()

        # Remove entry and exit regions
        n = len(signal)
        signal = signal[int(0.10*n):int(0.90*n)]

        if len(signal) < RAW_WINDOW:
            raise ValueError(
                f"C{cutter}, cut {cut_id}: recording too short"
            )

        # Four evenly spaced, non-overlapping windows
        starts = np.linspace(
            0,
            len(signal) - RAW_WINDOW,
            WINDOWS_PER_CUT,
            dtype=int
        )

        for start in starts:

            segment = signal[
                start:start + RAW_WINDOW
            ]

            # Anti-alias filtering + downsampling
            reduced = resample_poly(
                segment,
                up=1,
                down=DOWNSAMPLE,
                axis=0
            ).astype(np.float32)

            assert reduced.shape == (SEQ_LEN, 7)

            X_all.append(reduced)
            y_all.append(wear.loc[cut_id, "wear_max"])
            cutter_all.append(cutter)
            cut_all.append(cut_id)

        if cut_id % 50 == 0:
            print(
                f"  Completed {cut_id}/315",
                flush=True
            )

X = np.stack(X_all)
y = np.asarray(y_all, dtype=np.float32)
cutters = np.asarray(cutter_all, dtype=np.int16)
cuts = np.asarray(cut_all, dtype=np.int16)

assert X.shape == (3780, 2048, 7)
assert y.shape == (3780,)
assert np.isfinite(X).all()
assert np.isfinite(y).all()

# Raw amplitudes are preserved here.
# Normalization will be learned from training cutters only.

np.savez(
    OUT,
    X=X,
    y=y,
    cutter=cutters,
    cut=cuts
)

print("\nSEQUENCE PREPARATION COMPLETE")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Cutter counts:",
      dict(zip(*np.unique(cutters, return_counts=True))))
print("Saved to:", OUT)


Processing C1...
  Completed 50/315
  Completed 100/315
  Completed 150/315
  Completed 200/315
  Completed 250/315
  Completed 300/315

Processing C4...
  Completed 50/315
  Completed 100/315
  Completed 150/315
  Completed 200/315
  Completed 250/315
  Completed 300/315

Processing C6...
  Completed 50/315
  Completed 100/315
  Completed 150/315
  Completed 200/315
  Completed 250/315
  Completed 300/315

SEQUENCE PREPARATION COMPLETE
X shape: (3780, 2048, 7)
y shape: (3780,)
Cutter counts: {np.int16(1): np.int64(1260), np.int16(4): np.int64(1260), np.int16(6): np.int64(1260)}
Saved to: /kaggle/working/phm_sequences.npz


In [4]:
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ---------------------------------
# 1. Reproducibility
# ---------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

# ---------------------------------
# 2. Load prepared sequences
# ---------------------------------
data_path = Path("/kaggle/working/phm_sequences.npz")
assert data_path.exists(), "Run sequence preparation first."

with np.load(data_path) as data:
    X = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    cutters = data["cutter"].astype(int)
    cuts = data["cut"].astype(int)

assert X.shape == (3780, 2048, 7)
assert np.isfinite(X).all()
assert np.isfinite(y).all()

# ---------------------------------
# 3. Split by cutter and cut number
# ---------------------------------
test_cutter = 1

train_mask = (
    (cutters != test_cutter) &
    (cuts <= 220)
)
val_mask = (
    (cutters != test_cutter) &
    (cuts >= 221) & (cuts <= 260)
)
cal_mask = (
    (cutters != test_cutter) &
    (cuts >= 261)
)
test_mask = cutters == test_cutter

assert sum(train_mask) == 1760
assert sum(val_mask) == 320
assert sum(cal_mask) == 440
assert sum(test_mask) == 1260

print("Training windows:", sum(train_mask))
print("Validation windows:", sum(val_mask))
print("Reserved calibration windows:", sum(cal_mask))
print("Test windows:", sum(test_mask))

# ---------------------------------
# 4. Normalize using training only
# ---------------------------------
channel_mean = X[train_mask].mean(
    axis=(0, 1), dtype=np.float64
).astype(np.float32)

channel_std = X[train_mask].std(
    axis=(0, 1), dtype=np.float64
).astype(np.float32)

channel_std = np.maximum(channel_std, 1e-6)

y_mean = float(y[train_mask].mean())
y_std = max(float(y[train_mask].std()), 1e-6)

def prepare(mask):
    xx = (
        X[mask] - channel_mean
    ) / channel_std
    yy = (
        y[mask] - y_mean
    ) / y_std

    # PyTorch Conv1d expects [batch, channels, time]
    xx = np.transpose(xx, (0, 2, 1))
    return (
        torch.from_numpy(xx.copy()),
        torch.from_numpy(yy.astype(np.float32))
    )

X_train, y_train = prepare(train_mask)
X_val, y_val = prepare(val_mask)
X_test, y_test = prepare(test_mask)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)

# ---------------------------------
# 5. CNN + BiLSTM + Attention
# ---------------------------------
class CNNBiLSTMAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(7, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.lstm = nn.LSTM(
            input_size=64, hidden_size=64, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.attention = nn.Linear(128, 1)
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        features = self.cnn(x).transpose(1, 2)
        temporal, _ = self.lstm(features)
        weights = torch.softmax(self.attention(temporal), dim=1)
        context = (temporal * weights).sum(dim=1)
        return self.head(context).squeeze(-1)

cnn_model = CNNBiLSTMAttention().to(device)
with torch.no_grad():
    check = cnn_model(torch.randn(2, 7, 2048, device=device))
assert check.shape == (2,)
print("CNN forward pass successful.")

# ---------------------------------
# 6. Training helpers
# ---------------------------------
def predict_loader(net, loader):
    net.eval()
    chunks = []
    with torch.no_grad():
        for xb, _ in loader:
            chunks.append(net(xb.to(device)).cpu().numpy())
    return np.concatenate(chunks) * y_std + y_mean


def cut_level_predictions(preds, targets, cutter_ids, cut_ids):
    frame = pd.DataFrame({
        "cutter": cutter_ids, "cut": cut_ids,
        "actual": targets, "predicted": preds
    })
    return frame.groupby(["cutter", "cut"], as_index=False)[
        ["actual", "predicted"]
    ].mean()


criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    cnn_model.parameters(), lr=0.001, weight_decay=1e-4
)
MAX_EPOCHS = 25
PATIENCE = 5
best_val_mae = float("inf")
best_epoch = 0
best_state = None
no_improvement = 0
history = []

# ---------------------------------
# 7. Train (validation includes separate cutter IDs)
# ---------------------------------
for epoch in range(1, MAX_EPOCHS + 1):
    cnn_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        predictions = cnn_model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(cnn_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)

    train_loss = total_loss / len(train_loader.dataset)
    val_preds = predict_loader(cnn_model, val_loader)
    val_cut = cut_level_predictions(
        val_preds, y[val_mask], cutters[val_mask], cuts[val_mask]
    )
    assert len(val_cut) == 80
    val_mae = mean_absolute_error(val_cut["actual"], val_cut["predicted"])
    history.append({
        "epoch": epoch, "train_loss": train_loss, "val_mae_um": val_mae
    })
    print(
        f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | "
        f"Val MAE: {val_mae:.3f} µm", flush=True
    )
    if val_mae < best_val_mae - 0.01:
        best_val_mae = val_mae
        best_epoch = epoch
        best_state = copy.deepcopy(cnn_model.state_dict())
        no_improvement = 0
    else:
        no_improvement += 1
    if no_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

# ---------------------------------
# 8. Evaluate C1 (exploratory test; not pristine holdout)
# ---------------------------------
assert best_state is not None
cnn_model.load_state_dict(best_state)
test_preds = predict_loader(cnn_model, test_loader)
test_cut = cut_level_predictions(
    test_preds, y[test_mask], cutters[test_mask], cuts[test_mask]
)
assert len(test_cut) == 315
mae = mean_absolute_error(test_cut["actual"], test_cut["predicted"])
rmse = np.sqrt(mean_squared_error(test_cut["actual"], test_cut["predicted"]))
r2 = r2_score(test_cut["actual"], test_cut["predicted"])
print("\nCORRECTED CNN-BiLSTM RESULTS (EXPLORATORY)")
print("Test cutter: C1")
print("Best epoch:", best_epoch)
print(f"Validation MAE: {best_val_mae:.3f} µm")
print(f"Test MAE: {mae:.3f} µm")
print(f"Test RMSE: {rmse:.3f} µm")
print(f"Test R2: {r2:.3f}")

# ---------------------------------
# 9. Save with NEW filenames (preserve original results)
# ---------------------------------
out = Path("/kaggle/working")
pd.DataFrame(history).to_csv(out / "cnn_fold1_corrected_history.csv", index=False)
test_cut.to_csv(out / "cnn_fold1_corrected_predictions.csv", index=False)
torch.save({
    "model_state": best_state,
    "channel_mean": channel_mean,
    "channel_std": channel_std,
    "y_mean": y_mean,
    "y_std": y_std,
    "best_epoch": best_epoch,
    "test_cutter": test_cutter,
    "seed": SEED,
    "selection_metric": "validation cut-level MAE grouped by cutter and cut",
}, out / "cnn_fold1_corrected_best.pt")
print("Saved corrected model and results under cnn_fold1_corrected_* filenames.")

Device: cuda
Training windows: 1760
Validation windows: 320
Reserved calibration windows: 440
Test windows: 1260
CNN forward pass successful.
Epoch 01 | Train loss: 0.2692 | Val MAE: 26.629 µm
Epoch 02 | Train loss: 0.0820 | Val MAE: 26.243 µm
Epoch 03 | Train loss: 0.0776 | Val MAE: 23.589 µm
Epoch 04 | Train loss: 0.0663 | Val MAE: 20.818 µm
Epoch 05 | Train loss: 0.0584 | Val MAE: 22.517 µm
Epoch 06 | Train loss: 0.0459 | Val MAE: 21.911 µm
Epoch 07 | Train loss: 0.0466 | Val MAE: 21.894 µm
Epoch 08 | Train loss: 0.0401 | Val MAE: 19.268 µm
Epoch 09 | Train loss: 0.0395 | Val MAE: 23.711 µm
Epoch 10 | Train loss: 0.0387 | Val MAE: 22.389 µm
Epoch 11 | Train loss: 0.0362 | Val MAE: 20.483 µm
Epoch 12 | Train loss: 0.0426 | Val MAE: 21.874 µm
Epoch 13 | Train loss: 0.0368 | Val MAE: 23.026 µm
Early stopping triggered.

CORRECTED CNN-BiLSTM RESULTS (EXPLORATORY)
Test cutter: C1
Best epoch: 8
Validation MAE: 19.268 µm
Test MAE: 19.909 µm
Test RMSE: 27.287 µm
Test R2: 0.035
Saved correct

In [5]:

import pandas as pd
from sklearn.metrics import mean_absolute_error

# The model currently contains the saved best-epoch weights.
val_preds = predict_loader(cnn_model, val_loader)

validation_df = pd.DataFrame({
    "cutter": cutters[val_mask],
    "cut": cuts[val_mask],
    "actual": y[val_mask],
    "predicted": val_preds
})

# Correct: aggregate by BOTH cutter and cut
validation_cut = (
    validation_df
    .groupby(["cutter", "cut"], as_index=False)
    .agg({
        "actual": "mean",
        "predicted": "mean"
    })
)

print("VALIDATION DIAGNOSTIC")
print("Window-level observations:", len(validation_df))
print("Unique cutter/cut pairs:", len(validation_cut))

print("\nCorrect validation MAE:")
print(
    round(
        mean_absolute_error(
            validation_cut["actual"],
            validation_cut["predicted"]
        ), 3
    )
)

print("\nMAE by validation cutter:")

for cutter in [4, 6]:
    subset = validation_cut[
        validation_cut["cutter"] == cutter
    ]

    mae = mean_absolute_error(
        subset["actual"],
        subset["predicted"]
    )

    print("C{}: {:.3f} µm".format(cutter, mae))

VALIDATION DIAGNOSTIC
Window-level observations: 320
Unique cutter/cut pairs: 80

Correct validation MAE:
19.268

MAE by validation cutter:
C4: 19.481 µm
C6: 19.054 µm


In [6]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load features from your saved Kaggle notebook
feature_path = (
    "/kaggle/input/notebooks/kaziakaid/"
    "phm2010-01-data-exploration/phm_features.csv"
)

df = pd.read_csv(feature_path)

print("Feature dataset loaded:", df.shape)

# Exclude IDs and target from model inputs
feature_cols = [
    col for col in df.columns
    if col not in ["cutter", "cut", "wear_max"]
]

# Match the CNN-BiLSTM training split
train = df[
    (df["cutter"].isin([4, 6])) &
    (df["cut"] <= 220)
]

# C1 is the held-out test cutter
test = df[df["cutter"] == 1]

print("Training cuts:", len(train))
print("Testing cuts:", len(test))

model = RandomForestRegressor(
    n_estimators=300,
    max_features="sqrt",
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model.fit(
    train[feature_cols],
    train["wear_max"]
)

pred = model.predict(test[feature_cols])

mae = mean_absolute_error(test["wear_max"], pred)
rmse = np.sqrt(
    mean_squared_error(test["wear_max"], pred)
)
r2 = r2_score(test["wear_max"], pred)

print("\nFAIR RANDOM FOREST COMPARISON")
print("Test cutter: C1")
print("MAE: {:.3f} µm".format(mae))
print("RMSE: {:.3f} µm".format(rmse))
print("R2: {:.3f}".format(r2))

# Preserve the results
pd.DataFrame([{
    "Model": "Random Forest",
    "Test_Cutter": "C1",
    "Training_Cuts": len(train),
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}]).to_csv(
    "/kaggle/working/rf_fair_comparison.csv",
    index=False
)

print("\nSaved: rf_fair_comparison.csv")

Feature dataset loaded: (945, 87)
Training cuts: 440
Testing cuts: 315

FAIR RANDOM FOREST COMPARISON
Test cutter: C1
MAE: 13.781 µm
RMSE: 17.547 µm
R2: 0.601

Saved: rf_fair_comparison.csv


In [7]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error

path = Path("/kaggle/working/cnn_fold1_predictions.csv")

assert path.exists(), (
    "CNN predictions not found. Attach the saved "
    "notebook output containing cnn_fold1_predictions.csv."
)

df_pred = pd.read_csv(path).sort_values("cut")

print("CNN PREDICTION DIAGNOSTIC")
print("Total cuts:", len(df_pred))

print("\nActual wear range:")
print(
    df_pred["actual"].min(),
    df_pred["actual"].max()
)

print("\nPredicted wear range:")
print(
    df_pred["predicted"].min(),
    df_pred["predicted"].max()
)

print("\nPrediction bias:")
print(
    round(
        (df_pred["predicted"] - df_pred["actual"]).mean(),
        3
    ),
    "µm"
)

print("\nMAE by cutting stage:")

stages = [
    (1, 105, "Early"),
    (106, 210, "Middle"),
    (211, 315, "Late")
]

for start, end, stage in stages:
    subset = df_pred[
        (df_pred["cut"] >= start) &
        (df_pred["cut"] <= end)
    ]

    mae = mean_absolute_error(
        subset["actual"],
        subset["predicted"]
    )

    print(
        "{} stage: {:.3f} µm".format(stage, mae)
    )

plt.figure(figsize=(11, 5))

plt.plot(
    df_pred["cut"],
    df_pred["actual"],
    label="Actual wear"
)

plt.plot(
    df_pred["cut"],
    df_pred["predicted"],
    label="CNN-BiLSTM prediction"
)

plt.xlabel("Cut number")
plt.ylabel("Maximum tool wear (µm)")
plt.title("CNN-BiLSTM Prediction Diagnostic — C1")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

AssertionError: CNN predictions not found. Attach the saved notebook output containing cnn_fold1_predictions.csv.

In [ ]:

import torch
from pathlib import Path

checkpoint_path = Path(
    "/kaggle/working/cnn_fold1_best.pt"
)

assert checkpoint_path.exists(), (
    "CNN checkpoint not found. "
    "Check your saved Kaggle notebook outputs."
)

# Create a separate variable for the neural network
cnn_model = CNNBiLSTMAttention().to(device)

# Load saved model weights and normalization settings
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

cnn_model.load_state_dict(
    checkpoint["model_state"]
)

cnn_model.eval()

# Restore output normalization parameters
y_mean = checkpoint["y_mean"]
y_std = checkpoint["y_std"]

print("CNN-BiLSTM successfully restored.")
print("Best epoch:", checkpoint["best_epoch"])
print("Test cutter:", checkpoint["test_cutter"])
print("Model type:", type(cnn_model).__name__)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Uses the trained model and test_loader
# from the CNN-BiLSTM training cell.
window_preds = predict_loader(cnn_model, test_loader)

diagnostic = pd.DataFrame({
    "cutter": cutters[test_mask],
    "cut": cuts[test_mask],
    "actual": y[test_mask],
    "predicted": window_preds
})

# Calculate variation among the four windows per cut
summary = diagnostic.groupby(
    ["cutter", "cut"], as_index=False
).agg(
    actual=("actual", "mean"),
    predicted=("predicted", "mean"),
    window_std=("predicted", "std"),
    window_min=("predicted", "min"),
    window_max=("predicted", "max")
)

summary["absolute_error"] = abs(
    summary["predicted"] - summary["actual"]
)

print("WINDOW DISAGREEMENT DIAGNOSTIC")

print("\nMean within-cut prediction std:",
      round(summary["window_std"].mean(), 3))

print("Maximum within-cut prediction std:",
      round(summary["window_std"].max(), 3))

print("\nHighest-disagreement cuts:")
print(
    summary.nlargest(10, "window_std")[
        ["cut", "actual", "predicted",
         "window_std", "absolute_error"]
    ].round(3).to_string(index=False)
)

print("\nInspecting unusual prediction regions:")
print(
    summary[
        summary["cut"].between(15, 25) |
        summary["cut"].between(250, 260)
    ][
        ["cut", "actual", "predicted",
         "window_std", "absolute_error"]
    ].round(3).to_string(index=False)
)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    summary["cut"], summary["actual"],
    label="Actual wear"
)

ax.plot(
    summary["cut"], summary["predicted"],
    label="CNN prediction"
)

ax.fill_between(
    summary["cut"],
    summary["window_min"],
    summary["window_max"],
    alpha=0.2,
    label="Range across 4 windows"
)

ax.set_xlabel("Cut number")
ax.set_ylabel("Tool wear (µm)")
ax.set_title("C1: Variation Across Sensor Windows")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

summary.to_csv(
    "/kaggle/working/cnn_window_diagnostic.csv",
    index=False
)

In [ ]:

from pathlib import Path

folder = Path("/kaggle/working")

print("Available research files:\n")

for name in [
    "phm_sequences.npz",
    "cnn_fold1_corrected_best.pt",
    "cnn_fold1_corrected_history.csv",
    "cnn_fold1_corrected_predictions.csv",
    "rf_fair_comparison.csv"
]:
    path = folder / name
    print(name, "FOUND" if path.exists() else "MISSING")


In [ ]:

import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
DATA_PATH = Path("/kaggle/working/phm_sequences.npz")
SAVE_DIR = Path("/kaggle/working")

assert DATA_PATH.exists(), "Sequence file is missing."
assert DEVICE.type == "cuda", "Enable Kaggle GPU first."
assert "CNNBiLSTMAttention" in globals(), (
    "Run the corrected CNN training cell first "
    "to define the model architecture."
)

# Load the prepared signals
with np.load(DATA_PATH) as data:
    X_all = data["X"].astype(np.float32)
    y_all = data["y"].astype(np.float32)
    cutter_ids = data["cutter"].astype(int)
    cut_ids = data["cut"].astype(int)

assert X_all.shape == (3780, 2048, 7)
assert np.isfinite(X_all).all()
assert np.isfinite(y_all).all()

def train_one_fold(test_cutter, seed=42):
    print("\n" + "=" * 55, flush=True)
    print("TRAINING FOLD: C{}".format(test_cutter), flush=True)
    print("=" * 55, flush=True)

    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Split data
    train_mask = (
        (cutter_ids != test_cutter) &
        (cut_ids <= 220)
    )
    val_mask = (
        (cutter_ids != test_cutter) &
        (cut_ids >= 221) &
        (cut_ids <= 260)
    )
    cal_mask = (
        (cutter_ids != test_cutter) &
        (cut_ids >= 261)
    )
    test_mask = cutter_ids == test_cutter

    assert train_mask.sum() == 1760
    assert val_mask.sum() == 320
    assert cal_mask.sum() == 440
    assert test_mask.sum() == 1260

    print("Train:", train_mask.sum(), "windows")
    print("Validation:", val_mask.sum(), "windows")
    print("Calibration reserved:", cal_mask.sum(), "windows")
    print("Test:", test_mask.sum(), "windows")

    # Normalize using training data only
    channel_mean = X_all[train_mask].mean(
        axis=(0, 1), dtype=np.float64
    ).astype(np.float32)

    channel_std = X_all[train_mask].std(
        axis=(0, 1), dtype=np.float64
    ).astype(np.float32)

    channel_std = np.maximum(channel_std, 1e-6)

    y_mean = float(y_all[train_mask].mean())
    y_std = max(float(y_all[train_mask].std()), 1e-6)

    def make_loader(mask, batch_size, shuffle=False):
        xx = (
            X_all[mask] - channel_mean
        ) / channel_std

        xx = np.transpose(xx, (0, 2, 1))
        yy = (
            y_all[mask] - y_mean
        ) / y_std

        dataset = TensorDataset(
            torch.from_numpy(
                np.ascontiguousarray(xx, dtype=np.float32)
            ),
            torch.from_numpy(
                np.asarray(yy, dtype=np.float32)
            )
        )

        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=0
        )

    train_dl = make_loader(train_mask, 32, True)
    val_dl = make_loader(val_mask, 64)
    test_dl = make_loader(test_mask, 64)

    # Independent model for this fold
    cnn = CNNBiLSTMAttention().to(DEVICE)

    optimizer = torch.optim.AdamW(
        cnn.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )
    criterion = nn.MSELoss()

    def predict(dl):
        cnn.eval()
        outputs = []

        with torch.no_grad():
            for xb, _ in dl:
                xb = xb.to(DEVICE)
                pred = cnn(xb)
                outputs.append(pred.cpu().numpy())

        return (
            np.concatenate(outputs) * y_std + y_mean
        )

    def aggregate(predictions, mask):
        frame = pd.DataFrame({
            "cutter": cutter_ids[mask],
            "cut": cut_ids[mask],
            "actual": y_all[mask],
            "predicted": predictions
        })

        return (
            frame.groupby(
                ["cutter", "cut"], as_index=False
            )[["actual", "predicted"]]
            .mean()
        )

    best_mae = float("inf")
    best_epoch = 0
    best_state = None
    stale_epochs = 0
    history = []

    # Train
    for epoch in range(1, 26):
        cnn.train()
        total_loss = 0.0

        for xb, yb in train_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            pred = cnn(xb)
            loss = criterion(pred, yb)
            loss.backward()

            nn.utils.clip_grad_norm_(
                cnn.parameters(), 1.0
            )

            optimizer.step()
            total_loss += loss.item() * len(xb)

        train_loss = total_loss / len(train_dl.dataset)

        val_pred = predict(val_dl)
        val_cut = aggregate(val_pred, val_mask)

        assert len(val_cut) == 80

        val_mae = mean_absolute_error(
            val_cut["actual"],
            val_cut["predicted"]
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_mae_um": val_mae
        })

        print(
            "Epoch {:02d} | Loss {:.4f} | "
            "Val MAE {:.3f} µm".format(
                epoch, train_loss, val_mae
            ),
            flush=True
        )

        if val_mae < best_mae - 0.01:
            best_mae = val_mae
            best_epoch = epoch
            best_state = copy.deepcopy(
                cnn.state_dict()
            )
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= 5:
            print("Early stopping.", flush=True)
            break

    # Restore best weights
    assert best_state is not None
    cnn.load_state_dict(best_state)

    # Test at cut level
    test_pred = predict(test_dl)
    test_cut = aggregate(test_pred, test_mask)

    assert len(test_cut) == 315

    mae = mean_absolute_error(
        test_cut["actual"], test_cut["predicted"]
    )
    rmse = np.sqrt(mean_squared_error(
        test_cut["actual"], test_cut["predicted"]
    ))
    r2 = r2_score(
        test_cut["actual"], test_cut["predicted"]
    )

    print("\nRESULT C{}".format(test_cutter))
    print("Best epoch:", best_epoch)
    print("Validation MAE: {:.3f}".format(best_mae))
    print("Test MAE: {:.3f}".format(mae))
    print("Test RMSE: {:.3f}".format(rmse))
    print("Test R2: {:.3f}".format(r2))

    prefix = "cnn_fold_C{}".format(test_cutter)

    pd.DataFrame(history).to_csv(
        SAVE_DIR / (prefix + "_history.csv"),
        index=False
    )

    test_cut.to_csv(
        SAVE_DIR / (prefix + "_predictions.csv"),
        index=False
    )

    torch.save({
        "model_state": best_state,
        "channel_mean": channel_mean,
        "channel_std": channel_std,
        "y_mean": y_mean,
        "y_std": y_std,
        "seed": seed,
        "test_cutter": test_cutter,
        "best_epoch": best_epoch
    }, SAVE_DIR / (prefix + "_best.pt"))

    del cnn, train_dl, val_dl, test_dl
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "Test_Cutter": "C{}".format(test_cutter),
        "Best_Epoch": best_epoch,
        "Val_MAE": best_mae,
        "Test_MAE": mae,
        "Test_RMSE": rmse,
        "Test_R2": r2
    }

# Train the remaining two folds
remaining_results = []

for held_out in [4, 6]:
    result = train_one_fold(held_out, seed=42)
    remaining_results.append(result)

results_df = pd.DataFrame(remaining_results)

results_df.to_csv(
    SAVE_DIR / "cnn_remaining_folds_results.csv",
    index=False
)

print("\n" + "=" * 55)
print("REMAINING FOLDS COMPLETE")
print("=" * 55)
print(results_df.round(3).to_string(index=False))
print("\nAll checkpoints and predictions saved.")


In [ ]:

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load features from saved Kaggle input
FEATURE_PATH = Path(
    "/kaggle/input/notebooks/kaziakaid/"
    "phm2010-01-data-exploration/phm_features.csv"
)

assert FEATURE_PATH.exists(), "Feature CSV is missing."

rf_df = pd.read_csv(FEATURE_PATH)

feature_cols = [
    c for c in rf_df.columns
    if c not in ["cutter", "cut", "wear_max"]
]

assert len(rf_df) == 945
assert len(feature_cols) == 84
assert np.isfinite(
    rf_df[feature_cols + ["wear_max"]].to_numpy()
).all()

results = []

for held_out in [1, 4, 6]:

    # Same training cuts as the CNN experiments
    train_df = rf_df[
        (rf_df["cutter"] != held_out) &
        (rf_df["cut"] <= 220)
    ]

    test_df = rf_df[
        rf_df["cutter"] == held_out
    ].sort_values("cut")

    assert len(train_df) == 440
    assert len(test_df) == 315

    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        train_df[feature_cols],
        train_df["wear_max"]
    )

    predictions = rf_model.predict(
        test_df[feature_cols]
    )

    mae = mean_absolute_error(
        test_df["wear_max"], predictions
    )

    rmse = np.sqrt(mean_squared_error(
        test_df["wear_max"], predictions
    ))

    r2 = r2_score(
        test_df["wear_max"], predictions
    )

    results.append({
        "Test_Cutter": f"C{held_out}",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    pd.DataFrame({
        "cutter": held_out,
        "cut": test_df["cut"].to_numpy(),
        "actual": test_df["wear_max"].to_numpy(),
        "predicted": predictions
    }).to_csv(
        f"/kaggle/working/rf_matched_C{held_out}_predictions.csv",
        index=False
    )

results_df = pd.DataFrame(results)

print("\nMATCHED RANDOM FOREST RESULTS")
print(results_df.round(3).to_string(index=False))

print("\nMACRO AVERAGES")
print(
    results_df[["MAE", "RMSE", "R2"]]
    .mean().round(3)
)

results_df.to_csv(
    "/kaggle/working/rf_matched_all_folds.csv",
    index=False
)

print("\nSaved: rf_matched_all_folds.csv")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error

folder = Path("/kaggle/working")

rows = []

for cutter in [1, 4, 6]:
    # Random Forest predictions
    rf_path = folder / f"rf_matched_C{cutter}_predictions.csv"

    # Corrected CNN predictions
    if cutter == 1:
        cnn_path = folder / "cnn_fold1_corrected_predictions.csv"
    else:
        cnn_path = folder / f"cnn_fold_C{cutter}_predictions.csv"

    assert rf_path.exists(), f"Missing {rf_path}"
    assert cnn_path.exists(), f"Missing {cnn_path}"

    for name, path in [
        ("Random Forest", rf_path),
        ("CNN-BiLSTM", cnn_path)
    ]:
        data = pd.read_csv(path)

        assert len(data) == 315
        assert set(data["cut"]) == set(range(1, 316))
        assert data["cut"].is_unique

        for start, end, stage in [
            (1, 105, "Early"),
            (106, 210, "Middle"),
            (211, 315, "Late")
        ]:
            subset = data[
                data["cut"].between(start, end)
            ]

            mae = mean_absolute_error(
                subset["actual"],
                subset["predicted"]
            )

            bias = (
                subset["predicted"] -
                subset["actual"]
            ).mean()

            rows.append({
                "Cutter": f"C{cutter}",
                "Model": name,
                "Stage": stage,
                "MAE_um": mae,
                "Bias_um": bias
            })

results = pd.DataFrame(rows)

print("\nERROR BY CUTTING STAGE")
print(results.round(3).to_string(index=False))

print("\nOVERALL MAE BY MODEL")
print(
    results.groupby("Model")["MAE_um"]
    .mean().round(3)
)

results.to_csv(
    folder / "stage_error_comparison.csv",
    index=False
)

print("\nSaved: stage_error_comparison.csv")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

data_path = Path("/kaggle/working/phm_sequences.npz")

with np.load(data_path) as data:
    cutter_ids = data["cutter"].astype(int)
    cut_ids = data["cut"].astype(int)
    wear = data["y"].astype(float)

# 15 blocks per cutter, each containing 21 cuts
block_ids = ((cut_ids - 1) // 21) + 1

VAL_BLOCKS = [4, 12]
CAL_BLOCKS = [2, 8, 14]

records = []

for test_cutter in [1, 4, 6]:
    train_cutter_mask = cutter_ids != test_cutter

    train = (
        train_cutter_mask &
        ~np.isin(block_ids, VAL_BLOCKS + CAL_BLOCKS)
    )

    val = (
        train_cutter_mask &
        np.isin(block_ids, VAL_BLOCKS)
    )

    cal = (
        train_cutter_mask &
        np.isin(block_ids, CAL_BLOCKS)
    )

    test = cutter_ids == test_cutter

    assert (train | val | cal | test).all()
    assert (
        train.astype(int) +
        val.astype(int) +
        cal.astype(int) +
        test.astype(int)
    ).max() == 1

    print("\nHELD-OUT CUTTER C{}".format(test_cutter))

    for name, mask in [
        ("Train", train),
        ("Validation", val),
        ("Calibration", cal),
        ("Test", test)
    ]:
        pairs = set(zip(
            cutter_ids[mask],
            cut_ids[mask]
        ))

        print(
            name,
            "| cuts:", len(pairs),
            "| windows:", int(mask.sum())
        )

        # No test-label statistics are used
        # to design or evaluate this split.
        if name != "Test":
            print(
                "  Wear range:",
                round(wear[mask].min(), 2),
                "to",
                round(wear[mask].max(), 2),
                "µm"
            )

        for cutter, cut, block in zip(
            cutter_ids[mask],
            cut_ids[mask],
            block_ids[mask]
        ):
            records.append({
                "test_cutter": test_cutter,
                "cutter": int(cutter),
                "cut": int(cut),
                "block": int(block),
                "split": name
            })

splits = (
    pd.DataFrame(records)
    .drop_duplicates()
    .sort_values(["test_cutter", "cutter", "cut"])
)

output = Path("/kaggle/working/phm_protocol_v2_splits.csv")
splits.to_csv(output, index=False)

print("\nPROTOCOL V2 SAVED:", output)


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Input files
FEATURE_PATH = Path(
    "/kaggle/input/notebooks/kaziakaid/"
    "phm2010-01-data-exploration/phm_features.csv"
)

SPLIT_PATH = Path(
    "/kaggle/working/phm_protocol_v2_splits.csv"
)

OUT = Path("/kaggle/working")

assert FEATURE_PATH.exists(), "Feature dataset missing"
assert SPLIT_PATH.exists(), "Protocol V2 split file missing"

features = pd.read_csv(FEATURE_PATH)
splits = pd.read_csv(SPLIT_PATH)

feature_cols = [
    c for c in features.columns
    if c not in ["cutter", "cut", "wear_max"]
]

assert features.shape == (945, 87)
assert len(feature_cols) == 84

results = []

for test_cutter in [1, 4, 6]:

    # Get the exact Protocol V2 allocation
    fold_splits = splits[
        splits["test_cutter"] == test_cutter
    ][["cutter", "cut", "split"]]

    assert len(fold_splits) == 945
    assert not fold_splits.duplicated(
        ["cutter", "cut"]
    ).any()

    fold_data = features.merge(
        fold_splits,
        on=["cutter", "cut"],
        how="inner",
        validate="one_to_one"
    )

    assert len(fold_data) == 945

    train = fold_data[
        fold_data["split"] == "Train"
    ]

    test = fold_data[
        fold_data["split"] == "Test"
    ].sort_values("cut")

    assert len(train) == 420
    assert len(test) == 315
    assert set(test["cutter"]) == {test_cutter}

    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        train[feature_cols],
        train["wear_max"]
    )

    prediction = rf_model.predict(
        test[feature_cols]
    )

    mae = mean_absolute_error(
        test["wear_max"], prediction
    )

    rmse = np.sqrt(mean_squared_error(
        test["wear_max"], prediction
    ))

    r2 = r2_score(
        test["wear_max"], prediction
    )

    results.append({
        "Test_Cutter": f"C{test_cutter}",
        "Model": "Random Forest V2",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    pd.DataFrame({
        "cutter": test_cutter,
        "cut": test["cut"].to_numpy(),
        "actual": test["wear_max"].to_numpy(),
        "predicted": prediction
    }).to_csv(
        OUT / f"rf_v2_C{test_cutter}_predictions.csv",
        index=False
    )

    print(
        f"C{test_cutter} | "
        f"MAE={mae:.3f} | "
        f"RMSE={rmse:.3f} | "
        f"R2={r2:.3f}",
        flush=True
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    OUT / "rf_protocol_v2_results.csv",
    index=False
)

print("\nPROTOCOL V2 RANDOM FOREST RESULTS")
print(results_df.round(3).to_string(index=False))

print("\nMACRO AVERAGE")
print(
    results_df[["MAE", "RMSE", "R2"]]
    .mean().round(3)
)

print("\nSaved: rf_protocol_v2_results.csv")


In [ ]:

import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# 1. CONFIGURATION
# ==========================================
SEED = 42
EPOCHS = 25
PATIENCE = 5
BATCH_SIZE = 32

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

WORK = Path("/kaggle/working")
SEQ_PATH = WORK / "phm_sequences.npz"
SPLIT_PATH = WORK / "phm_protocol_v2_splits.csv"

assert DEVICE.type == "cuda", "Enable GPU."
assert SEQ_PATH.exists(), "Sequence file missing."
assert SPLIT_PATH.exists(), "V2 split file missing."

print("Device:", DEVICE)

with np.load(SEQ_PATH) as data:
    X_v2 = data["X"].astype(np.float32)
    y_v2 = data["y"].astype(np.float32)
    cutter_v2 = data["cutter"].astype(int)
    cut_v2 = data["cut"].astype(int)

splits_v2 = pd.read_csv(SPLIT_PATH)

assert X_v2.shape == (3780, 2048, 7)
assert np.isfinite(X_v2).all()
assert np.isfinite(y_v2).all()

# ==========================================
# 2. DEFINE MODEL
# ==========================================
class V2CNNBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(7, 32, 7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(128, 1)

        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.transpose(1, 2)
        x, _ = self.lstm(x)

        scores = self.attention(x)
        weights = torch.softmax(scores, dim=1)

        context = (x * weights).sum(dim=1)
        return self.head(context).squeeze(-1)


# ==========================================
# 3. TRAIN ONE FOLD
# ==========================================
def train_v2_fold(test_cutter):

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    print("\n" + "=" * 50)
    print(f"PROTOCOL V2 | TEST CUTTER C{test_cutter}")
    print("=" * 50, flush=True)

    fold_split = splits_v2[
        splits_v2["test_cutter"] == test_cutter
    ].copy()

    assert len(fold_split) == 945
    assert not fold_split.duplicated(
        ["cutter", "cut"]
    ).any()

    # Map each window to its split, preserving
    # all four windows from the same cut.
    window_meta = pd.DataFrame({
        "cutter": cutter_v2,
        "cut": cut_v2
    })

    window_meta = window_meta.merge(
        fold_split[["cutter", "cut", "split"]],
        on=["cutter", "cut"],
        how="left",
        validate="many_to_one",
        sort=False
    )

    assert window_meta["split"].notna().all()

    masks = {
        name: (
            window_meta["split"].to_numpy() == name
        )
        for name in [
            "Train", "Validation",
            "Calibration", "Test"
        ]
    }

    tr = masks["Train"]
    va = masks["Validation"]
    ca = masks["Calibration"]
    te = masks["Test"]

    assert tr.sum() == 1680
    assert va.sum() == 336
    assert ca.sum() == 504
    assert te.sum() == 1260
    assert set(cutter_v2[te]) == {test_cutter}

    print(
        "Windows:",
        {name: int(mask.sum())
         for name, mask in masks.items()}
    )

    # Normalize using training partition only
    channel_mean = X_v2[tr].mean(
        axis=(0, 1), dtype=np.float64
    ).astype(np.float32)

    channel_std = X_v2[tr].std(
        axis=(0, 1), dtype=np.float64
    ).astype(np.float32)

    channel_std = np.maximum(channel_std, 1e-6)

    target_mean = float(y_v2[tr].mean())
    target_std = max(float(y_v2[tr].std()), 1e-6)

    def loader(mask, shuffle=False):
        x = (
            X_v2[mask] - channel_mean
        ) / channel_std

        x = np.transpose(x, (0, 2, 1))
        y = (y_v2[mask] - target_mean) / target_std

        dataset = TensorDataset(
            torch.from_numpy(
                np.ascontiguousarray(x, dtype=np.float32)
            ),
            torch.from_numpy(
                np.asarray(y, dtype=np.float32)
            )
        )

        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=0
        )

    train_dl = loader(tr, shuffle=True)
    val_dl = loader(va)
    test_dl = loader(te)

    cnn_v2 = V2CNNBiLSTM().to(DEVICE)

    optimizer = torch.optim.AdamW(
        cnn_v2.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    loss_fn = nn.MSELoss()

    def predict(dl):
        cnn_v2.eval()
        output = []

        with torch.no_grad():
            for xb, _ in dl:
                pred = cnn_v2(xb.to(DEVICE))
                output.append(pred.cpu().numpy())

        return (
            np.concatenate(output) * target_std
            + target_mean
        )

    def aggregate(predictions, mask):
        frame = pd.DataFrame({
            "cutter": cutter_v2[mask],
            "cut": cut_v2[mask],
            "actual": y_v2[mask],
            "predicted": predictions
        })

        return frame.groupby(
            ["cutter", "cut"], as_index=False
        )[["actual", "predicted"]].mean()

    best_mae = float("inf")
    best_epoch = 0
    best_state = None
    stale = 0
    history = []

    # ======================================
    # 4. TRAINING
    # ======================================
    for epoch in range(1, EPOCHS + 1):
        cnn_v2.train()
        total_loss = 0.0

        for xb, yb in train_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            pred = cnn_v2(xb)

            loss = loss_fn(pred, yb)
            loss.backward()

            nn.utils.clip_grad_norm_(
                cnn_v2.parameters(), 1.0
            )

            optimizer.step()
            total_loss += loss.item() * len(xb)

        train_loss = total_loss / len(train_dl.dataset)

        val_preds = predict(val_dl)
        val_cuts = aggregate(val_preds, va)

        assert len(val_cuts) == 84

        val_mae = mean_absolute_error(
            val_cuts["actual"],
            val_cuts["predicted"]
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_mae": val_mae
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Loss {train_loss:.4f} | "
            f"Val MAE {val_mae:.3f} µm",
            flush=True
        )

        if val_mae < best_mae - 0.01:
            best_mae = val_mae
            best_epoch = epoch
            best_state = copy.deepcopy(
                cnn_v2.state_dict()
            )
            stale = 0
        else:
            stale += 1

        if stale >= PATIENCE:
            print("Early stopping.")
            break

    assert best_state is not None
    cnn_v2.load_state_dict(best_state)

    # ======================================
    # 5. HELD-OUT TEST EVALUATION
    # ======================================
    test_preds = predict(test_dl)
    test_cuts = aggregate(test_preds, te)

    assert len(test_cuts) == 315

    mae = mean_absolute_error(
        test_cuts["actual"], test_cuts["predicted"]
    )
    rmse = np.sqrt(mean_squared_error(
        test_cuts["actual"], test_cuts["predicted"]
    ))
    r2 = r2_score(
        test_cuts["actual"], test_cuts["predicted"]
    )

    print(f"\nRESULT C{test_cutter}")
    print("Best epoch:", best_epoch)
    print(f"Validation MAE: {best_mae:.3f}")
    print(f"Test MAE: {mae:.3f}")
    print(f"Test RMSE: {rmse:.3f}")
    print(f"Test R2: {r2:.3f}")

    # ======================================
    # 6. SAVE RESULTS
    # ======================================
    prefix = f"cnn_v2_C{test_cutter}"

    pd.DataFrame(history).to_csv(
        WORK / f"{prefix}_history.csv",
        index=False
    )

    test_cuts.to_csv(
        WORK / f"{prefix}_predictions.csv",
        index=False
    )

    torch.save({
        "model_state": best_state,
        "channel_mean": channel_mean,
        "channel_std": channel_std,
        "target_mean": target_mean,
        "target_std": target_std,
        "test_cutter": test_cutter,
        "best_epoch": best_epoch,
        "seed": SEED,
        "protocol": "V2"
    }, WORK / f"{prefix}_best.pt")

    del cnn_v2, train_dl, val_dl, test_dl
    torch.cuda.empty_cache()

    return {
        "Cutter": f"C{test_cutter}",
        "Best_Epoch": best_epoch,
        "Validation_MAE": best_mae,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# ==========================================
# 7. RUN ALL THREE FOLDS
# ==========================================
all_results = []

for cutter in [1, 4, 6]:
    result = train_v2_fold(cutter)
    all_results.append(result)

final_results = pd.DataFrame(all_results)

final_results.to_csv(
    WORK / "cnn_protocol_v2_results.csv",
    index=False
)

print("\n" + "=" * 50)
print("CNN-BiLSTM PROTOCOL V2 RESULTS")
print("=" * 50)
print(final_results.round(3).to_string(index=False))

print("\nMACRO AVERAGE")
print(
    final_results[["MAE", "RMSE", "R2"]]
    .mean().round(3)
)

print("\nAll V2 CNN results saved.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import torch

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals(), (
    "Run the V2 CNN model-definition cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    cutters = data["cutter"].astype(int)
    cuts = data["cut"].astype(int)

splits = pd.read_csv(
    WORK / "phm_protocol_v2_splits.csv"
)

def predict_cuts(model, mask, checkpoint):
    indices = np.where(mask)[0]

    mean = checkpoint["channel_mean"]
    std = checkpoint["channel_std"]

    outputs = []

    model.eval()

    with torch.no_grad():
        for start in range(0, len(indices), 64):
            batch_ids = indices[start:start + 64]

            batch = (X[batch_ids] - mean) / std
            batch = np.transpose(batch, (0, 2, 1))
            batch = np.ascontiguousarray(batch)

            tensor = torch.from_numpy(batch).to(DEVICE)

            pred = model(tensor).cpu().numpy()
            pred = (
                pred * checkpoint["target_std"]
                + checkpoint["target_mean"]
            )

            outputs.append(pred)

    window_preds = np.concatenate(outputs)

    frame = pd.DataFrame({
        "cutter": cutters[indices],
        "cut": cuts[indices],
        "actual": y[indices],
        "predicted": window_preds
    })

    return frame.groupby(
        ["cutter", "cut"], as_index=False
    )[["actual", "predicted"]].mean()

results = []

for held_out in [1, 4, 6]:

    print(f"\nEVALUATING C{held_out}", flush=True)

    checkpoint = torch.load(
        WORK / f"cnn_v2_C{held_out}_best.pt",
        map_location="cpu",
        weights_only=False
    )

    model = V2CNNBiLSTM().to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])

    fold = splits[
        splits["test_cutter"] == held_out
    ]

    split_lookup = {
        (int(r.cutter), int(r.cut)): r.split
        for r in fold.itertuples(index=False)
    }

    labels = np.array([
        split_lookup[(int(c), int(k))]
        for c, k in zip(cutters, cuts)
    ])

    cal_mask = labels == "Calibration"
    test_mask = labels == "Test"

    calibration = predict_cuts(
        model, cal_mask, checkpoint
    )
    test = predict_cuts(
        model, test_mask, checkpoint
    )

    assert len(calibration) == 126
    assert len(test) == 315

    # Absolute residuals, one per calibration cut
    residuals = np.abs(
        calibration["actual"].to_numpy()
        - calibration["predicted"].to_numpy()
    )

    alpha = 0.10
    n = len(residuals)

    # Finite-sample split-conformal quantile
    rank = int(np.ceil((n + 1) * (1 - alpha)))
    q = np.sort(residuals)[rank - 1]

    test["lower"] = test["predicted"] - q
    test["upper"] = test["predicted"] + q

    covered = (
        (test["actual"] >= test["lower"]) &
        (test["actual"] <= test["upper"])
    )

    coverage = float(covered.mean())
    mean_width = float(
        (test["upper"] - test["lower"]).mean()
    )

    cal_mae = float(residuals.mean())
    test_mae = float(np.abs(
        test["actual"] - test["predicted"]
    ).mean())

    print(f"Calibration MAE: {cal_mae:.3f} µm")
    print(f"Test MAE: {test_mae:.3f} µm")
    print(f"Conformal threshold: {q:.3f} µm")
    print(f"Test coverage: {coverage * 100:.2f}%")
    print(f"Mean interval width: {mean_width:.3f} µm")

    results.append({
        "Cutter": f"C{held_out}",
        "Calibration_MAE": cal_mae,
        "Test_MAE": test_mae,
        "Conformal_q": q,
        "Coverage": coverage,
        "Mean_Width": mean_width
    })

    test.to_csv(
        WORK / f"cnn_v2_C{held_out}_intervals.csv",
        index=False
    )

    del model

results_df = pd.DataFrame(results)

print("\nCONFORMAL COVERAGE RESULTS")
print(results_df.round(3).to_string(index=False))

results_df.to_csv(
    WORK / "cnn_v2_conformal_results.csv",
    index=False
)

print("\nSaved conformal results.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

WORK = Path("/kaggle/working")

rows = []

for cutter in [1, 4, 6]:
    path = WORK / f"cnn_v2_C{cutter}_intervals.csv"

    assert path.exists(), f"Missing {path.name}"

    df = pd.read_csv(path).sort_values("cut")

    assert len(df) == 315
    assert df["cut"].is_unique
    assert df[[
        "actual", "predicted", "lower", "upper"
    ]].notna().all().all()

    for start, end, stage in [
        (1, 105, "Early"),
        (106, 210, "Middle"),
        (211, 315, "Late")
    ]:
        part = df[df["cut"].between(start, end)]

        covered = (
            (part["actual"] >= part["lower"]) &
            (part["actual"] <= part["upper"])
        )

        above_upper = (
            part["actual"] > part["upper"]
        )

        below_lower = (
            part["actual"] < part["lower"]
        )

        rows.append({
            "Cutter": f"C{cutter}",
            "Stage": stage,
            "Coverage_pct": covered.mean() * 100,
            "Above_upper_pct": above_upper.mean() * 100,
            "Below_lower_pct": below_lower.mean() * 100,
            "MAE_um": np.abs(
                part["actual"] - part["predicted"]
            ).mean(),
            "Bias_um": (
                part["predicted"] - part["actual"]
            ).mean()
        })

results = pd.DataFrame(rows)

print("\nCONFORMAL COVERAGE BY WEAR STAGE")
print(results.round(2).to_string(index=False))

results.to_csv(
    WORK / "conformal_stage_analysis.csv",
    index=False
)

print("\nSaved: conformal_stage_analysis.csv")


In [ ]:

import copy
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

WORK = Path("/kaggle/working")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [42, 43, 44, 45, 46]

assert DEVICE.type == "cuda", "Enable the Kaggle GPU."
assert "V2CNNBiLSTM" in globals(), "Run the V2 CNN definition first."

with np.load(WORK / "phm_sequences.npz") as data:
    X_ens = data["X"].astype(np.float32)
    y_ens = data["y"].astype(np.float32)
    cutter_ens = data["cutter"].astype(int)
    cut_ens = data["cut"].astype(int)

split_table = pd.read_csv(WORK / "phm_protocol_v2_splits.csv")

assert X_ens.shape == (3780, 2048, 7)
assert np.isfinite(X_ens).all()

def train_ensemble_member(held_out, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    fold = split_table[
        split_table["test_cutter"] == held_out
    ][["cutter", "cut", "split"]]

    metadata = pd.DataFrame({
        "cutter": cutter_ens,
        "cut": cut_ens,
        "row_id": np.arange(len(cut_ens))
    })

    metadata = metadata.merge(
        fold,
        on=["cutter", "cut"],
        how="left",
        validate="many_to_one",
        sort=False
    ).sort_values("row_id")

    assert metadata["split"].notna().all()

    masks = {
        part: metadata["split"].to_numpy() == part
        for part in ["Train", "Validation", "Calibration", "Test"]
    }

    tr = masks["Train"]
    va = masks["Validation"]
    ca = masks["Calibration"]
    te = masks["Test"]

    assert (tr.sum(), va.sum(), ca.sum(), te.sum()) == (
        1680, 336, 504, 1260
    )

    # Fit normalization using training data only.
    mean = X_ens[tr].mean(
        axis=(0, 1), dtype=np.float64
    ).astype(np.float32)

    std = X_ens[tr].std(
        axis=(0, 1), dtype=np.float64
    ).astype(np.float32)

    std = np.maximum(std, 1e-6)

    target_mean = float(y_ens[tr].mean())
    target_std = max(float(y_ens[tr].std()), 1e-6)

    def make_loader(mask, shuffle=False):
        xx = (X_ens[mask] - mean) / std
        xx = np.ascontiguousarray(
            xx.transpose(0, 2, 1), dtype=np.float32
        )
        yy = ((y_ens[mask] - target_mean) /
              target_std).astype(np.float32)

        return DataLoader(
            TensorDataset(
                torch.from_numpy(xx),
                torch.from_numpy(yy)
            ),
            batch_size=32 if shuffle else 64,
            shuffle=shuffle,
            num_workers=0
        )

    train_loader = make_loader(tr, True)
    val_loader = make_loader(va)
    cal_loader = make_loader(ca)
    test_loader = make_loader(te)

    net = V2CNNBiLSTM().to(DEVICE)
    optimizer = torch.optim.AdamW(
        net.parameters(), lr=0.001, weight_decay=1e-4
    )
    loss_fn = nn.MSELoss()

    def predict(loader):
        net.eval()
        chunks = []
        with torch.no_grad():
            for xb, _ in loader:
                pred = net(xb.to(DEVICE))
                chunks.append(pred.cpu().numpy())

        return (np.concatenate(chunks) * target_std
                + target_mean)

    def aggregate(pred, mask):
        frame = pd.DataFrame({
            "cutter": cutter_ens[mask],
            "cut": cut_ens[mask],
            "actual": y_ens[mask],
            "predicted": pred
        })
        return frame.groupby(
            ["cutter", "cut"], as_index=False
        )[["actual", "predicted"]].mean()

    best_mae = float("inf")
    best_epoch = 0
    best_state = None
    stale = 0

    for epoch in range(1, 26):
        net.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()

        val_pred = aggregate(predict(val_loader), va)

        val_mae = np.abs(
            val_pred["actual"] - val_pred["predicted"]
        ).mean()

        if val_mae < best_mae - 0.01:
            best_mae = float(val_mae)
            best_epoch = epoch
            best_state = copy.deepcopy(net.state_dict())
            stale = 0
        else:
            stale += 1

        if stale >= 5:
            break

    assert best_state is not None
    net.load_state_dict(best_state)

    # Save calibration and test predictions
    # at cut level, using the best checkpoint.
    calibration = aggregate(predict(cal_loader), ca)
    test = aggregate(predict(test_loader), te)

    assert len(calibration) == 126
    assert len(test) == 315

    prefix = f"ensemble_C{held_out}_seed{seed}"

    calibration.to_csv(
        WORK / f"{prefix}_calibration.csv",
        index=False
    )
    test.to_csv(
        WORK / f"{prefix}_test.csv",
        index=False
    )

    torch.save({
        "model_state": best_state,
        "channel_mean": mean,
        "channel_std": std,
        "target_mean": target_mean,
        "target_std": target_std,
        "seed": seed,
        "test_cutter": held_out,
        "best_epoch": best_epoch,
        "protocol": "V2"
    }, WORK / f"{prefix}.pt")

    print(
        f"C{held_out} | Seed {seed} | "
        f"Best epoch {best_epoch} | "
        f"Val MAE {best_mae:.3f}",
        flush=True
    )

    del net, optimizer
    del train_loader, val_loader, cal_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "Cutter": f"C{held_out}",
        "Seed": seed,
        "Best_Epoch": best_epoch,
        "Validation_MAE": best_mae
    }

# Train all 15 models.
records = []

for cutter in [1, 4, 6]:
    for seed in SEEDS:
        records.append(train_ensemble_member(cutter, seed))

        # Incremental checkpoint of run metadata.
        pd.DataFrame(records).to_csv(
            WORK / "ensemble_training_summary.csv",
            index=False
        )

print("\nENSEMBLE TRAINING COMPLETE")
print(pd.DataFrame(records).round(3).to_string(index=False))


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

WORK = Path("/kaggle/working")
SEEDS = [42, 43, 44, 45, 46]

def combine_ensemble_predictions(cutter, partition):
    """
    Combine saved predictions from five models.
    Match each prediction by cutter and cut ID.
    """

    merged = None

    for seed in SEEDS:
        path = WORK / (
            f"ensemble_C{cutter}_seed{seed}_{partition}.csv"
        )

        assert path.exists(), f"Missing: {path.name}"

        df = pd.read_csv(path)

        required = [
            "cutter", "cut", "actual", "predicted"
        ]
        assert all(c in df.columns for c in required)
        assert not df[["cutter", "cut"]].duplicated().any()
        assert np.isfinite(df[required].to_numpy()).all()

        df = df[required].rename(
            columns={
                "actual": f"actual_{seed}",
                "predicted": f"pred_seed{seed}"
            }
        )

        if merged is None:
            merged = df
        else:
            merged = merged.merge(
                df,
                on=["cutter", "cut"],
                how="inner",
                validate="one_to_one"
            )

    expected = 126 if partition == "calibration" else 315
    assert len(merged) == expected, (
        f"Expected {expected} cuts, got {len(merged)}"
    )

    # Verify that all five models have the same targets
    target_columns = [f"actual_{s}" for s in SEEDS]
    targets = merged[target_columns].to_numpy()

    assert np.allclose(
        targets,
        targets[:, [0]],
        rtol=0,
        atol=1e-4
    ), "Wear labels differ between models!"

    merged["actual"] = targets[:, 0]

    prediction_cols = [
        f"pred_seed{s}" for s in SEEDS
    ]

    predictions = merged[prediction_cols].to_numpy()

    merged["ensemble_mean"] = predictions.mean(axis=1)
    merged["ensemble_std"] = predictions.std(
        axis=1, ddof=1
    )

    merged["absolute_error"] = np.abs(
        merged["actual"] - merged["ensemble_mean"]
    )

    merged = merged.drop(
        columns=target_columns
    ).sort_values(["cutter", "cut"]).reset_index(drop=True)

    return merged


results = []

for cutter in [1, 4, 6]:
    print(f"\nEVALUATING ENSEMBLE C{cutter}")

    calibration = combine_ensemble_predictions(
        cutter, "calibration"
    )

    test = combine_ensemble_predictions(
        cutter, "test"
    )

    assert set(test["cutter"]) == {cutter}
    assert len(test) == 315

    mae = mean_absolute_error(
        test["actual"], test["ensemble_mean"]
    )

    rmse = np.sqrt(mean_squared_error(
        test["actual"], test["ensemble_mean"]
    ))

    r2 = r2_score(
        test["actual"], test["ensemble_mean"]
    )

    mean_std = test["ensemble_std"].mean()

    # Descriptive only: test error relationship
    rho = test[
        ["ensemble_std", "absolute_error"]
    ].corr(method="spearman").iloc[0, 1]

    print(f"Test MAE: {mae:.3f} µm")
    print(f"Test RMSE: {rmse:.3f} µm")
    print(f"Test R2: {r2:.3f}")
    print(f"Mean ensemble std: {mean_std:.3f} µm")
    print(f"Std-error Spearman rho: {rho:.3f}")

    results.append({
        "Cutter": f"C{cutter}",
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Mean_Std": mean_std,
        "Std_Error_Rho": rho
    })

    calibration.to_csv(
        WORK / f"ensemble_C{cutter}_calibration_combined.csv",
        index=False
    )

    test.to_csv(
        WORK / f"ensemble_C{cutter}_test_combined.csv",
        index=False
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    WORK / "ensemble_evaluation_results.csv",
    index=False
)

print("\n" + "=" * 55)
print("DEEP ENSEMBLE EVALUATION RESULTS")
print("=" * 55)

print(results_df.round(3).to_string(index=False))

print("\nMACRO AVERAGES")
print(
    results_df[["MAE", "RMSE", "R2", "Mean_Std"]]
    .mean().round(3)
)

print("\nAll ensemble evaluation files saved.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

ALPHA = 0.10
SCALE_FLOOR = 1.0  # Fixed, in micrometers

def conformal_quantile(scores, alpha=0.10):
    scores = np.asarray(scores, dtype=float)
    assert len(scores) > 0
    assert np.isfinite(scores).all()

    n = len(scores)
    rank = int(np.ceil((n + 1) * (1 - alpha)))

    if rank > n:
        return float("inf")

    return float(np.sort(scores)[rank - 1])


def evaluate_intervals(df, lower, upper):
    actual = df["actual"].to_numpy(dtype=float)

    covered = (actual >= lower) & (actual <= upper)
    width = upper - lower

    # Interval score: penalizes wide intervals and
    # observations falling outside their intervals.
    score = width.copy()
    score += (2 / ALPHA) * np.maximum(lower - actual, 0)
    score += (2 / ALPHA) * np.maximum(actual - upper, 0)

    return {
        "Coverage_pct": 100 * covered.mean(),
        "Mean_Width_um": width.mean(),
        "Interval_Score": score.mean()
    }


results = []

for cutter in [1, 4, 6]:
    cal_path = WORK / (
        f"ensemble_C{cutter}_calibration_combined.csv"
    )
    test_path = WORK / (
        f"ensemble_C{cutter}_test_combined.csv"
    )

    assert cal_path.exists(), f"Missing {cal_path.name}"
    assert test_path.exists(), f"Missing {test_path.name}"

    cal = pd.read_csv(cal_path)
    test = pd.read_csv(test_path)

    assert len(cal) == 126
    assert len(test) == 315

    for df in [cal, test]:
        assert not df[["cutter", "cut"]].duplicated().any()
        assert np.isfinite(
            df[[
                "actual",
                "ensemble_mean",
                "ensemble_std"
            ]].to_numpy()
        ).all()
        assert (df["ensemble_std"] >= 0).all()

    cal_actual = cal["actual"].to_numpy()
    cal_mean = cal["ensemble_mean"].to_numpy()

    test_mean = test["ensemble_mean"].to_numpy()

    # =====================================
    # METHOD A: Standard split conformal
    # =====================================
    cal_residuals = np.abs(cal_actual - cal_mean)
    q_standard = conformal_quantile(cal_residuals)

    standard_lower = test_mean - q_standard
    standard_upper = test_mean + q_standard

    standard_metrics = evaluate_intervals(
        test, standard_lower, standard_upper
    )

    # =====================================
    # METHOD B: Adaptive split conformal
    # =====================================
    cal_scale = (
        cal["ensemble_std"].to_numpy()
        + SCALE_FLOOR
    )

    test_scale = (
        test["ensemble_std"].to_numpy()
        + SCALE_FLOOR
    )

    # Normalized calibration residuals
    normalized_scores = cal_residuals / cal_scale
    q_adaptive = conformal_quantile(normalized_scores)

    # Vary interval width per test cut
    half_width = q_adaptive * test_scale

    adaptive_lower = test_mean - half_width
    adaptive_upper = test_mean + half_width

    adaptive_metrics = evaluate_intervals(
        test, adaptive_lower, adaptive_upper
    )

    # =====================================
    # SAVE BOTH METHODS
    # =====================================
    output = test[[
        "cutter", "cut", "actual",
        "ensemble_mean", "ensemble_std",
        "absolute_error"
    ]].copy()

    output["standard_lower"] = standard_lower
    output["standard_upper"] = standard_upper
    output["adaptive_lower"] = adaptive_lower
    output["adaptive_upper"] = adaptive_upper

    output.to_csv(
        WORK / f"ensemble_C{cutter}_conformal_comparison.csv",
        index=False
    )

    for method, q, metrics in [
        ("Standard", q_standard, standard_metrics),
        ("Adaptive", q_adaptive, adaptive_metrics)
    ]:
        results.append({
            "Cutter": f"C{cutter}",
            "Method": method,
            "Quantile": q,
            **metrics
        })

    print(f"\nC{cutter}")
    print(
        f"Standard: coverage "
        f"{standard_metrics['Coverage_pct']:.2f}% | "
        f"width {standard_metrics['Mean_Width_um']:.2f} µm"
    )
    print(
        f"Adaptive: coverage "
        f"{adaptive_metrics['Coverage_pct']:.2f}% | "
        f"width {adaptive_metrics['Mean_Width_um']:.2f} µm"
    )

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("ENSEMBLE CONFORMAL COMPARISON")
print("=" * 60)
print(results_df.round(3).to_string(index=False))

print("\nMACRO AVERAGES BY METHOD")
print(
    results_df.groupby("Method")[
        ["Coverage_pct", "Mean_Width_um", "Interval_Score"]
    ].mean().round(3).to_string()
)

results_df.to_csv(
    WORK / "ensemble_conformal_comparison.csv",
    index=False
)

print("\nAll comparison files saved.")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import spearmanr

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals(), (
    "Run the V2 model definition cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X_ig = data["X"].astype(np.float32)
    cutter_ig = data["cutter"].astype(int)
    cut_ig = data["cut"].astype(int)

assert X_ig.shape == (3780, 2048, 7)

SENSOR_NAMES = [
    "Sensor_1", "Sensor_2", "Sensor_3",
    "Sensor_4", "Sensor_5", "Sensor_6",
    "Sensor_7"
]

# Predefined cuts distributed across tool life
SELECTED_CUTS = np.linspace(
    1, 315, 15, dtype=int
)

IG_STEPS = 12
NOISE_LEVEL = 0.01


# ==========================================
# 1. INTEGRATED GRADIENTS FUNCTION
# ==========================================

def integrated_gradients_channels(model, x):
    """
    Input:
        x: normalized tensor [1, 7, 2048]

    Baseline:
        Zero tensor = training channel means.

    Output:
        Normalized absolute IG importance
        for seven sensor channels.
    """

    baseline = torch.zeros_like(x)
    difference = x - baseline
    gradient_sum = torch.zeros_like(x)

    # Keep BatchNorm and Dropout in evaluation mode.
    model.eval()

    # Disable cuDNN during gradient computation.
    # Required for LSTM backward in eval mode.
    with torch.backends.cudnn.flags(enabled=False):

        for step in range(1, IG_STEPS + 1):
            alpha = step / IG_STEPS

            interpolated = (
                baseline + alpha * difference
            ).detach().requires_grad_(True)

            prediction = model(interpolated)

            gradient = torch.autograd.grad(
                outputs=prediction.sum(),
                inputs=interpolated,
                create_graph=False
            )[0]

            gradient_sum += gradient.detach()

    attribution = difference * (
        gradient_sum / IG_STEPS
    )

    importance = (
        attribution.abs()
        .sum(dim=2)
        .squeeze(0)
        .detach()
        .cpu()
        .numpy()
    )

    importance = importance / (
        importance.sum() + 1e-12
    )

    return importance


# ==========================================
# 2. COSINE SIMILARITY
# ==========================================

def cosine_similarity(a, b):
    denominator = (
        np.linalg.norm(a) *
        np.linalg.norm(b)
    )

    if denominator <= 1e-12:
        return np.nan

    return float(np.dot(a, b) / denominator)


# ==========================================
# 3. MAIN EXPERIMENT
# ==========================================

records = []
importance_records = []

for held_out in [1, 4, 6]:

    print(
        f"\nIG EXPLANATION STABILITY: C{held_out}",
        flush=True
    )

    checkpoint = torch.load(
        WORK / f"cnn_v2_C{held_out}_best.pt",
        map_location="cpu",
        weights_only=False
    )

    # IMPORTANT: This must be INSIDE the loop.
    model = V2CNNBiLSTM().to(DEVICE)

    model.load_state_dict(
        checkpoint["model_state"]
    )

    model.eval()

    mean = np.asarray(
        checkpoint["channel_mean"],
        dtype=np.float32
    )

    std = np.asarray(
        checkpoint["channel_std"],
        dtype=np.float32
    )

    ensemble = pd.read_csv(
        WORK / f"ensemble_C{held_out}_test_combined.csv"
    ).set_index("cut")

    rng = np.random.default_rng(
        2027 + held_out
    )

    for cut_number in SELECTED_CUTS:

        indices = np.where(
            (cutter_ig == held_out) &
            (cut_ig == cut_number)
        )[0]

        assert len(indices) == 4

        # Select first prepared window consistently.
        raw = X_ig[indices[0]]

        normalized = (raw - mean) / std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.T[None, :, :],
                dtype=np.float32
            )
        ).to(DEVICE)

        noise = rng.normal(
            loc=0.0,
            scale=NOISE_LEVEL,
            size=tuple(x.shape)
        ).astype(np.float32)

        x_perturbed = x + torch.from_numpy(
            noise
        ).to(DEVICE)

        # Calculate Integrated Gradients.
        original_ig = integrated_gradients_channels(
            model, x
        )

        perturbed_ig = integrated_gradients_channels(
            model, x_perturbed
        )

        similarity = cosine_similarity(
            original_ig,
            perturbed_ig
        )

        # Predictions do not need gradients.
        with torch.no_grad():

            original_prediction = float(
                model(x).item() *
                checkpoint["target_std"] +
                checkpoint["target_mean"]
            )

            perturbed_prediction = float(
                model(x_perturbed).item() *
                checkpoint["target_std"] +
                checkpoint["target_mean"]
            )

        ref = ensemble.loc[cut_number]

        records.append({
            "Cutter": f"C{held_out}",
            "Cut": int(cut_number),
            "IG_Cosine": similarity,
            "IG_Instability": (
                1 - similarity
                if np.isfinite(similarity)
                else np.nan
            ),
            "Prediction_Change_um": abs(
                original_prediction -
                perturbed_prediction
            ),
            "Ensemble_Std_um": float(
                ref["ensemble_std"]
            ),
            "Ensemble_Abs_Error_um": float(
                ref["absolute_error"]
            )
        })

        for channel_idx, sensor in enumerate(
            SENSOR_NAMES
        ):

            importance_records.append({
                "Cutter": f"C{held_out}",
                "Cut": int(cut_number),
                "Sensor": sensor,
                "IG_Original": float(
                    original_ig[channel_idx]
                ),
                "IG_Perturbed": float(
                    perturbed_ig[channel_idx]
                )
            })

    del model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ==========================================
# 4. SAVE RESULTS
# ==========================================

results = pd.DataFrame(records)

importance_df = pd.DataFrame(
    importance_records
)

assert len(results) == 45
assert len(importance_df) == 315

results.to_csv(
    WORK / "ig_stability_pilot.csv",
    index=False
)

importance_df.to_csv(
    WORK / "ig_channel_importance_pilot.csv",
    index=False
)


# ==========================================
# 5. PRINT RESULTS
# ==========================================

print("\nIG STABILITY PILOT RESULTS")

summary = results.groupby("Cutter").agg(
    Mean_IG_Cosine=("IG_Cosine", "mean"),
    Mean_Instability=("IG_Instability", "mean"),
    Mean_Prediction_Change_um=(
        "Prediction_Change_um", "mean"
    )
)

print(summary.round(5).to_string())


print("\nUNCERTAINTY VS IG INSTABILITY")

for cutter in ["C1", "C4", "C6"]:

    part = results[
        results["Cutter"] == cutter
    ].dropna()

    if (
        part["Ensemble_Std_um"].nunique() > 1
        and part["IG_Instability"].nunique() > 1
    ):

        rho, p_value = spearmanr(
            part["Ensemble_Std_um"],
            part["IG_Instability"]
        )

        print(
            cutter,
            "| Spearman rho:",
            round(float(rho), 3),
            "| exploratory p:",
            round(float(p_value), 3)
        )

    else:
        print(
            cutter,
            "| insufficient variation"
        )


print("\nSaved:")
print("ig_stability_pilot.csv")
print("ig_channel_importance_pilot.csv")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

stability = pd.read_csv(
    WORK / "ig_stability_pilot.csv"
)

importance = pd.read_csv(
    WORK / "ig_channel_importance_pilot.csv"
)

assert len(stability) == 45
assert len(importance) == 315

pd.set_option("display.precision", 12)

print("\n1. IG INSTABILITY AT HIGH PRECISION")

for cutter, group in stability.groupby("Cutter"):

    values = group["IG_Instability"].to_numpy()

    print(f"\n{cutter}")
    print(f"Minimum: {values.min():.12e}")
    print(f"Median:  {np.median(values):.12e}")
    print(f"Mean:    {values.mean():.12e}")
    print(f"Maximum: {values.max():.12e}")
    print(f"Exact zeros: {(values == 0).sum()} / {len(values)}")

print("\n2. CHANNEL IMPORTANCE CHANGES")

importance["Absolute_Change"] = np.abs(
    importance["IG_Original"] -
    importance["IG_Perturbed"]
)

changes = (
    importance.groupby(["Cutter", "Cut"])
    ["Absolute_Change"]
    .sum()
    .reset_index(name="Total_Channel_Change")
)

for cutter, group in changes.groupby("Cutter"):
    print(f"\n{cutter}")
    print(
        "Mean channel change:",
        f"{group['Total_Channel_Change'].mean():.12e}"
    )
    print(
        "Maximum channel change:",
        f"{group['Total_Channel_Change'].max():.12e}"
    )

print("\n3. PREDICTION SENSITIVITY")

print(
    stability.groupby("Cutter")[
        "Prediction_Change_um"
    ].agg(["mean", "median", "max"])
    .to_string()
)

print("\n4. SENSOR IMPORTANCE DISTRIBUTION")

importance_summary = (
    importance.groupby(["Cutter", "Sensor"])
    ["IG_Original"]
    .mean()
    .reset_index()
)

print(importance_summary.to_string(index=False))

changes.to_csv(
    WORK / "ig_channel_change_diagnostic.csv",
    index=False
)

print("\nSaved: ig_channel_change_diagnostic.csv")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals(), (
    "Run the V2 CNN model definition cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X = data["X"].astype(np.float32)
    cutter_ids = data["cutter"].astype(int)
    cut_ids = data["cut"].astype(int)

assert X.shape == (3780, 2048, 7)

SELECTED_CUTS = [1, 79, 158, 237, 315]
NOISE_LEVELS = [0.01, 0.05, 0.10]
REPETITIONS = 3
IG_STEPS = 16

# ==========================================
# 1. INTEGRATED GRADIENTS
# ==========================================

def calculate_ig(model, x):
    """
    Return full signed IG attribution and
    completeness error.

    Input shape: [1, 7, 2048]
    """

    model.eval()

    baseline = torch.zeros_like(x)
    difference = x - baseline
    gradient_sum = torch.zeros_like(x)

    # Disable cuDNN to allow LSTM gradients
    # while preserving evaluation behavior.
    with torch.backends.cudnn.flags(enabled=False):

        with torch.enable_grad():
            # Trapezoidal integration: endpoints
            # have half weight.
            for step in range(IG_STEPS + 1):
                alpha = step / IG_STEPS

                interpolated = (
                    baseline + alpha * difference
                ).detach().requires_grad_(True)

                prediction = model(interpolated)

                gradient = torch.autograd.grad(
                    prediction.sum(),
                    interpolated
                )[0]

                weight = (
                    0.5
                    if step in [0, IG_STEPS]
                    else 1.0
                )

                gradient_sum += (
                    weight * gradient.detach()
                )

            attribution = (
                difference * gradient_sum / IG_STEPS
            )

            with torch.no_grad():
                prediction_difference = (
                    model(x) - model(baseline)
                ).item()

    attribution_sum = attribution.sum().item()

    # Normalized-target units, not micrometers.
    completeness_error = abs(
        attribution_sum - prediction_difference
    )

    return (
        attribution.squeeze(0).detach().cpu().numpy(),
        completeness_error,
        prediction_difference
    )


def safe_cosine(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()

    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom < 1e-12:
        return np.nan

    return float(
        np.clip(np.dot(a, b) / denom, -1, 1)
    )


def channel_importance(attribution):
    # Shape: [7, 2048]
    importance = np.abs(attribution).sum(axis=1)
    return importance / max(
        importance.sum(), 1e-12
    )


# ==========================================
# 2. RUN SENSITIVITY EXPERIMENT
# ==========================================

rows = []
completeness_rows = []

for held_out in [1, 4, 6]:

    print(
        f"\nIG SENSITIVITY: C{held_out}",
        flush=True
    )

    checkpoint = torch.load(
        WORK / f"cnn_v2_C{held_out}_best.pt",
        map_location="cpu",
        weights_only=False
    )

    model = V2CNNBiLSTM().to(DEVICE)
    model.load_state_dict(
        checkpoint["model_state"]
    )
    model.eval()

    mean = np.asarray(
        checkpoint["channel_mean"],
        dtype=np.float32
    )
    std = np.asarray(
        checkpoint["channel_std"],
        dtype=np.float32
    )

    target_std = float(checkpoint["target_std"])
    rng = np.random.default_rng(9000 + held_out)

    for cut_number in SELECTED_CUTS:

        idx = np.where(
            (cutter_ids == held_out) &
            (cut_ids == cut_number)
        )[0]

        assert len(idx) == 4

        # First saved sensor window, consistently.
        raw = X[idx[0]]
        normalized = (raw - mean) / std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.T[None],
                dtype=np.float32
            )
        ).to(DEVICE)

        original_ig, original_error, original_delta = (
            calculate_ig(model, x)
        )

        importance_original = channel_importance(
            original_ig
        )

        completeness_rows.append({
            "Cutter": f"C{held_out}",
            "Cut": cut_number,
            "Completeness_Error_um": (
                original_error * target_std
            ),
            "Prediction_Difference_um": (
                abs(original_delta) * target_std
            )
        })

        with torch.no_grad():
            original_pred = model(x).item()

        for noise_level in NOISE_LEVELS:
            for repetition in range(REPETITIONS):

                noise = rng.normal(
                    0,
                    noise_level,
                    size=tuple(x.shape)
                ).astype(np.float32)

                perturbed = x + torch.from_numpy(
                    noise
                ).to(DEVICE)

                perturbed_ig, error, _ = (
                    calculate_ig(model, perturbed)
                )

                importance_perturbed = (
                    channel_importance(perturbed_ig)
                )

                with torch.no_grad():
                    new_pred = model(
                        perturbed
                    ).item()

                full_cosine = safe_cosine(
                    original_ig,
                    perturbed_ig
                )

                channel_cosine = safe_cosine(
                    importance_original,
                    importance_perturbed
                )

                channel_l1 = np.abs(
                    importance_original -
                    importance_perturbed
                ).sum()

                rows.append({
                    "Cutter": f"C{held_out}",
                    "Cut": cut_number,
                    "Noise_Level": noise_level,
                    "Repetition": repetition + 1,
                    "Full_IG_Cosine": full_cosine,
                    "Channel_Cosine": channel_cosine,
                    "Channel_L1": channel_l1,
                    "Prediction_Change_um": (
                        abs(new_pred - original_pred)
                        * target_std
                    ),
                    "IG_Completeness_Error_um": (
                        error * target_std
                    )
                })

    del model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ==========================================
# 3. SUMMARIZE RESULTS
# ==========================================

results = pd.DataFrame(rows)
completeness = pd.DataFrame(completeness_rows)

assert len(results) == 135
assert len(completeness) == 15
assert results.notna().all().all()

results.to_csv(
    WORK / "ig_sensitivity_validation.csv",
    index=False
)

completeness.to_csv(
    WORK / "ig_completeness_validation.csv",
    index=False
)

summary = results.groupby(
    ["Cutter", "Noise_Level"]
).agg(
    Full_IG_Cosine=("Full_IG_Cosine", "mean"),
    Channel_Cosine=("Channel_Cosine", "mean"),
    Channel_L1=("Channel_L1", "mean"),
    Prediction_Change_um=(
        "Prediction_Change_um", "mean"
    ),
    Completeness_Error_um=(
        "IG_Completeness_Error_um", "mean"
    )
).reset_index()

print("\nIG SENSITIVITY VALIDATION RESULTS")
print(summary.round(6).to_string(index=False))

print("\nORIGINAL IG COMPLETENESS CHECK")
print(
    completeness.groupby("Cutter")[
        "Completeness_Error_um"
    ].agg(["mean", "max"])
    .round(6)
    .to_string()
)

print("\nSaved:")
print("ig_sensitivity_validation.csv")
print("ig_completeness_validation.csv")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals(), (
    "Run the V2 model definition cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X_conv = data["X"].astype(np.float32)
    cutter_conv = data["cutter"].astype(int)
    cut_conv = data["cut"].astype(int)

STEP_COUNTS = [16, 32, 64, 128]
CHECK_CUT = 158

def ig_completeness_test(model, x, steps):
    model.eval()
    baseline = torch.zeros_like(x)
    difference = x - baseline
    gradient_sum = torch.zeros_like(x)

    with torch.backends.cudnn.flags(enabled=False):
        with torch.enable_grad():
            for k in range(steps + 1):
                alpha = k / steps

                sample = (
                    baseline + alpha * difference
                ).detach().requires_grad_(True)

                output = model(sample)

                grad = torch.autograd.grad(
                    output.sum(), sample
                )[0]

                weight = 0.5 if k in (0, steps) else 1.0
                gradient_sum += weight * grad.detach()

            attribution = (
                difference * gradient_sum / steps
            )

            with torch.no_grad():
                delta = (
                    model(x) - model(baseline)
                ).item()

    attribution_sum = attribution.sum().item()
    error = abs(attribution_sum - delta)

    relative_error = (
        error / abs(delta)
        if abs(delta) > 1e-8
        else np.nan
    )

    return {
        "Attribution_Sum": attribution_sum,
        "Output_Difference": delta,
        "Abs_Error_Normalized": error,
        "Relative_Error_pct": relative_error * 100
    }

records = []

for held_out in [1, 4, 6]:
    checkpoint = torch.load(
        WORK / f"cnn_v2_C{held_out}_best.pt",
        map_location="cpu",
        weights_only=False
    )

    net = V2CNNBiLSTM().to(DEVICE)
    net.load_state_dict(checkpoint["model_state"])
    net.eval()

    idx = np.where(
        (cutter_conv == held_out) &
        (cut_conv == CHECK_CUT)
    )[0]

    assert len(idx) == 4

    raw = X_conv[idx[0]]
    mean = np.asarray(
        checkpoint["channel_mean"],
        dtype=np.float32
    )
    std = np.asarray(
        checkpoint["channel_std"],
        dtype=np.float32
    )

    normalized = (raw - mean) / std

    x = torch.from_numpy(
        np.ascontiguousarray(
            normalized.T[None],
            dtype=np.float32
        )
    ).to(DEVICE)

    print(f"\nCHECKING C{held_out}, CUT {CHECK_CUT}",
          flush=True)

    for steps in STEP_COUNTS:
        result = ig_completeness_test(
            net, x, steps
        )

        records.append({
            "Cutter": f"C{held_out}",
            "Cut": CHECK_CUT,
            "Steps": steps,
            "Abs_Error_um": (
                result["Abs_Error_Normalized"] *
                checkpoint["target_std"]
            ),
            "Relative_Error_pct": (
                result["Relative_Error_pct"]
            ),
            "Output_Difference_um": (
                result["Output_Difference"] *
                checkpoint["target_std"]
            )
        })

        print(
            f"Steps {steps:3d} | "
            f"Error {records[-1]['Abs_Error_um']:.6f} µm | "
            f"Relative {records[-1]['Relative_Error_pct']:.4f}%",
            flush=True
        )

    del net

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

convergence = pd.DataFrame(records)

print("\nIG CONVERGENCE RESULTS")
print(convergence.round(6).to_string(index=False))

convergence.to_csv(
    WORK / "ig_convergence_results.csv",
    index=False
)

print("\nSaved: ig_convergence_results.csv")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "ig_completeness_test" in globals(), (
    "Run the previous IG convergence cell first."
)
assert "V2CNNBiLSTM" in globals()

with np.load(WORK / "phm_sequences.npz") as data:
    X_audit = data["X"].astype(np.float32)
    cutter_audit = data["cutter"].astype(int)
    cut_audit = data["cut"].astype(int)

AUDIT_CUTS = [1, 158, 315]
AUDIT_STEPS = [32, 128]

rows = []

for cutter in [1, 4, 6]:
    print(f"\nIG CONVERGENCE AUDIT: C{cutter}", flush=True)

    checkpoint = torch.load(
        WORK / f"cnn_v2_C{cutter}_best.pt",
        map_location="cpu",
        weights_only=False
    )

    model = V2CNNBiLSTM().to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    mean = np.asarray(
        checkpoint["channel_mean"], dtype=np.float32
    )
    std = np.asarray(
        checkpoint["channel_std"], dtype=np.float32
    )
    target_std = float(checkpoint["target_std"])

    for cut_number in AUDIT_CUTS:
        indices = np.where(
            (cutter_audit == cutter) &
            (cut_audit == cut_number)
        )[0]

        assert len(indices) == 4

        raw = X_audit[indices[0]]
        normalized = (raw - mean) / std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.T[None], dtype=np.float32
            )
        ).to(DEVICE)

        for steps in AUDIT_STEPS:
            result = ig_completeness_test(
                model, x, steps
            )

            absolute_error_um = (
                result["Abs_Error_Normalized"] *
                target_std
            )

            delta_um = (
                result["Output_Difference"] *
                target_std
            )

            relative_error = result["Relative_Error_pct"]

            # Relative error can be misleading if
            # the output difference is near zero.
            valid_relative = abs(delta_um) >= 1.0

            rows.append({
                "Cutter": f"C{cutter}",
                "Cut": cut_number,
                "Steps": steps,
                "Abs_Error_um": absolute_error_um,
                "Output_Difference_um": delta_um,
                "Relative_Error_pct": relative_error,
                "Relative_Metric_Valid": valid_relative,
                "Pass_1pct": (
                    bool(relative_error < 1.0)
                    if valid_relative else False
                )
            })

            print(
                f"Cut {cut_number:3d} | "
                f"Steps {steps:3d} | "
                f"Abs error {absolute_error_um:.5f} µm | "
                f"Relative {relative_error:.4f}% | "
                f"Valid: {valid_relative}",
                flush=True
            )

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

audit = pd.DataFrame(rows)

audit.to_csv(
    WORK / "ig_convergence_multicut_audit.csv",
    index=False
)

print("\nIG MULTICUT CONVERGENCE RESULTS")
print(
    audit[
        ["Cutter", "Cut", "Steps",
         "Abs_Error_um", "Relative_Error_pct",
         "Relative_Metric_Valid"]
    ].round(5).to_string(index=False)
)

print("\nSUMMARY BY STEP COUNT")

for steps in AUDIT_STEPS:
    part = audit[audit["Steps"] == steps]
    valid = part[part["Relative_Metric_Valid"]]

    print(f"\n{steps} integration steps")
    print("Evaluated cuts:", len(part))
    print("Valid relative comparisons:", len(valid))
    print(
        "Maximum absolute error:",
        round(part["Abs_Error_um"].max(), 6),
        "µm"
    )

    if len(valid) > 0:
        print(
            "Median relative error:",
            round(valid["Relative_Error_pct"].median(), 5),
            "%"
        )
        print(
            "Maximum relative error:",
            round(valid["Relative_Error_pct"].max(), 5),
            "%"
        )
        print(
            "Below 1%:",
            int(valid["Pass_1pct"].sum()),
            "/",
            len(valid)
        )

print("\nSaved: ig_convergence_multicut_audit.csv")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals(), (
    "Run the V2 CNN model-definition cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X_cutig = data["X"].astype(np.float32)
    cutter_cutig = data["cutter"].astype(int)
    cut_cutig = data["cut"].astype(int)

assert X_cutig.shape == (3780, 2048, 7)

AUDIT_CUTS = [1, 158, 315]
STEPS = 128
NOISE = 0.05

def cut_integrated_gradients(net, x, steps=128):
    """
    x contains ALL four windows:
        [4, 7, 2048]

    The prediction being explained is
    the mean output across four windows.

    Attributions retain all four windows,
    so their sum corresponds to that mean.
    """
    net.eval()

    baseline = torch.zeros_like(x)
    difference = x - baseline
    gradient_sum = torch.zeros_like(x)

    with torch.backends.cudnn.flags(enabled=False):
        with torch.enable_grad():
            for k in range(steps + 1):
                alpha = k / steps

                sample = (
                    baseline + alpha * difference
                ).detach().requires_grad_(True)

                # The cut-level prediction
                output = net(sample).mean()

                grad = torch.autograd.grad(
                    output, sample
                )[0]

                weight = (
                    0.5 if k in (0, steps)
                    else 1.0
                )

                gradient_sum += (
                    weight * grad.detach()
                )

            attribution = (
                difference * gradient_sum / steps
            )

            with torch.no_grad():
                delta = (
                    net(x).mean() -
                    net(baseline).mean()
                ).item()

    completeness = abs(
        attribution.sum().item() - delta
    )

    return (
        attribution.detach().cpu().numpy(),
        completeness,
        delta
    )


def stable_cosine(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()

    denominator = (
        np.linalg.norm(a) *
        np.linalg.norm(b)
    )

    if denominator <= 1e-12:
        return np.nan

    return float(
        np.clip(np.dot(a, b) / denominator, -1, 1)
    )


def importance_by_channel(attr):
    # attr has shape [4, 7, 2048]
    importance = np.abs(attr).sum(axis=(0, 2))

    return importance / max(
        importance.sum(), 1e-12
    )


rows = []

for held_out in [1, 4, 6]:
    print(
        f"\nCUT-LEVEL IG: C{held_out}",
        flush=True
    )

    checkpoint = torch.load(
        WORK / f"ensemble_C{held_out}_seed42.pt",
        map_location="cpu",
        weights_only=False
    )

    net = V2CNNBiLSTM().to(DEVICE)
    net.load_state_dict(
        checkpoint["model_state"]
    )
    net.eval()

    mean = np.asarray(
        checkpoint["channel_mean"],
        dtype=np.float32
    )
    std = np.asarray(
        checkpoint["channel_std"],
        dtype=np.float32
    )

    target_std = float(checkpoint["target_std"])
    target_mean = float(checkpoint["target_mean"])

    saved = pd.read_csv(
        WORK / f"ensemble_C{held_out}_test_combined.csv"
    ).set_index("cut")

    rng = np.random.default_rng(
        5000 + held_out
    )

    for cut_number in AUDIT_CUTS:
        indices = np.where(
            (cutter_cutig == held_out) &
            (cut_cutig == cut_number)
        )[0]

        assert len(indices) == 4

        # Keep all four windows.
        raw = X_cutig[indices]

        normalized = (raw - mean) / std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.transpose(0, 2, 1),
                dtype=np.float32
            )
        ).to(DEVICE)

        noise = rng.normal(
            0, NOISE, size=tuple(x.shape)
        ).astype(np.float32)

        perturbed = x + torch.from_numpy(
            noise
        ).to(DEVICE)

        original_attr, err1, delta1 = (
            cut_integrated_gradients(net, x, STEPS)
        )

        perturbed_attr, err2, delta2 = (
            cut_integrated_gradients(
                net, perturbed, STEPS
            )
        )

        with torch.no_grad():
            prediction = (
                net(x).mean().item()
                * target_std + target_mean
            )

            perturbed_prediction = (
                net(perturbed).mean().item()
                * target_std + target_mean
            )

        # Verify the explanation model matches
        # the saved seed-42 prediction.
        saved_prediction = float(
            saved.loc[
                cut_number, "pred_seed42"
            ]
        )

        assert np.isclose(
            prediction, saved_prediction,
            atol=0.05, rtol=0
        ), (
            f"Checkpoint prediction mismatch "
            f"for C{held_out} cut {cut_number}"
        )

        original_channel = (
            importance_by_channel(original_attr)
        )

        perturbed_channel = (
            importance_by_channel(perturbed_attr)
        )

        full_cosine = stable_cosine(
            original_attr, perturbed_attr
        )

        channel_cosine = stable_cosine(
            original_channel, perturbed_channel
        )

        rows.append({
            "Cutter": f"C{held_out}",
            "Cut": cut_number,
            "Full_IG_Cosine": full_cosine,
            "Channel_Cosine": channel_cosine,
            "Channel_L1": float(
                np.abs(
                    original_channel -
                    perturbed_channel
                ).sum()
            ),
            "Prediction_Change_um": abs(
                prediction - perturbed_prediction
            ),
            "Original_Completeness_um": (
                err1 * target_std
            ),
            "Perturbed_Completeness_um": (
                err2 * target_std
            ),
            "Original_Delta_um": (
                delta1 * target_std
            ),
            "Perturbed_Delta_um": (
                delta2 * target_std
            ),
            "Ensemble_Std_um": float(
                saved.loc[
                    cut_number, "ensemble_std"
                ]
            ),
            "Ensemble_Abs_Error_um": float(
                saved.loc[
                    cut_number, "absolute_error"
                ]
            )
        })

    del net

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


results = pd.DataFrame(rows)

assert len(results) == 9
assert results.notna().all().all()

output = WORK / "ig_cut_level_pilot.csv"
results.to_csv(output, index=False)

print("\nCUT-LEVEL IG PILOT RESULTS")

columns = [
    "Cutter",
    "Cut",
    "Full_IG_Cosine",
    "Channel_Cosine",
    "Channel_L1",
    "Prediction_Change_um",
    "Original_Completeness_um",
    "Perturbed_Completeness_um",
    "Ensemble_Std_um"
]

print(
    results[columns]
    .round(6)
    .to_string(index=False)
)

print("\nSaved: ig_cut_level_pilot.csv")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals(), (
    "Run the V2 CNN model-definition cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X_cutig = data["X"].astype(np.float32)
    cutter_cutig = data["cutter"].astype(int)
    cut_cutig = data["cut"].astype(int)

assert X_cutig.shape == (3780, 2048, 7)

AUDIT_CUTS = [1, 158, 315]
STEPS = 128
NOISE = 0.05

def cut_integrated_gradients(net, x, steps=128):
    """
    x contains ALL four windows:
        [4, 7, 2048]

    The prediction being explained is
    the mean output across four windows.

    Attributions retain all four windows,
    so their sum corresponds to that mean.
    """
    net.eval()

    baseline = torch.zeros_like(x)
    difference = x - baseline
    gradient_sum = torch.zeros_like(x)

    with torch.backends.cudnn.flags(enabled=False):
        with torch.enable_grad():
            for k in range(steps + 1):
                alpha = k / steps

                sample = (
                    baseline + alpha * difference
                ).detach().requires_grad_(True)

                # The cut-level prediction
                output = net(sample).mean()

                grad = torch.autograd.grad(
                    output, sample
                )[0]

                weight = (
                    0.5 if k in (0, steps)
                    else 1.0
                )

                gradient_sum += (
                    weight * grad.detach()
                )

            attribution = (
                difference * gradient_sum / steps
            )

            with torch.no_grad():
                delta = (
                    net(x).mean() -
                    net(baseline).mean()
                ).item()

    completeness = abs(
        attribution.sum().item() - delta
    )

    return (
        attribution.detach().cpu().numpy(),
        completeness,
        delta
    )


def stable_cosine(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()

    denominator = (
        np.linalg.norm(a) *
        np.linalg.norm(b)
    )

    if denominator <= 1e-12:
        return np.nan

    return float(
        np.clip(np.dot(a, b) / denominator, -1, 1)
    )


def importance_by_channel(attr):
    # attr has shape [4, 7, 2048]
    importance = np.abs(attr).sum(axis=(0, 2))

    return importance / max(
        importance.sum(), 1e-12
    )


rows = []

for held_out in [1, 4, 6]:
    print(
        f"\nCUT-LEVEL IG: C{held_out}",
        flush=True
    )

    checkpoint = torch.load(
        WORK / f"ensemble_C{held_out}_seed42.pt",
        map_location="cpu",
        weights_only=False
    )

    net = V2CNNBiLSTM().to(DEVICE)
    net.load_state_dict(
        checkpoint["model_state"]
    )
    net.eval()

    mean = np.asarray(
        checkpoint["channel_mean"],
        dtype=np.float32
    )
    std = np.asarray(
        checkpoint["channel_std"],
        dtype=np.float32
    )

    target_std = float(checkpoint["target_std"])
    target_mean = float(checkpoint["target_mean"])

    saved = pd.read_csv(
        WORK / f"ensemble_C{held_out}_test_combined.csv"
    ).set_index("cut")

    rng = np.random.default_rng(
        5000 + held_out
    )

    for cut_number in AUDIT_CUTS:
        indices = np.where(
            (cutter_cutig == held_out) &
            (cut_cutig == cut_number)
        )[0]

        assert len(indices) == 4

        # Keep all four windows.
        raw = X_cutig[indices]

        normalized = (raw - mean) / std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.transpose(0, 2, 1),
                dtype=np.float32
            )
        ).to(DEVICE)

        noise = rng.normal(
            0, NOISE, size=tuple(x.shape)
        ).astype(np.float32)

        perturbed = x + torch.from_numpy(
            noise
        ).to(DEVICE)

        original_attr, err1, delta1 = (
            cut_integrated_gradients(net, x, STEPS)
        )

        perturbed_attr, err2, delta2 = (
            cut_integrated_gradients(
                net, perturbed, STEPS
            )
        )

        with torch.no_grad():
            prediction = (
                net(x).mean().item()
                * target_std + target_mean
            )

            perturbed_prediction = (
                net(perturbed).mean().item()
                * target_std + target_mean
            )

        # Verify the explanation model matches
        # the saved seed-42 prediction.
        saved_prediction = float(
            saved.loc[
                cut_number, "pred_seed42"
            ]
        )

        assert np.isclose(
            prediction, saved_prediction,
            atol=0.05, rtol=0
        ), (
            f"Checkpoint prediction mismatch "
            f"for C{held_out} cut {cut_number}"
        )

        original_channel = (
            importance_by_channel(original_attr)
        )

        perturbed_channel = (
            importance_by_channel(perturbed_attr)
        )

        full_cosine = stable_cosine(
            original_attr, perturbed_attr
        )

        channel_cosine = stable_cosine(
            original_channel, perturbed_channel
        )

        rows.append({
            "Cutter": f"C{held_out}",
            "Cut": cut_number,
            "Full_IG_Cosine": full_cosine,
            "Channel_Cosine": channel_cosine,
            "Channel_L1": float(
                np.abs(
                    original_channel -
                    perturbed_channel
                ).sum()
            ),
            "Prediction_Change_um": abs(
                prediction - perturbed_prediction
            ),
            "Original_Completeness_um": (
                err1 * target_std
            ),
            "Perturbed_Completeness_um": (
                err2 * target_std
            ),
            "Original_Delta_um": (
                delta1 * target_std
            ),
            "Perturbed_Delta_um": (
                delta2 * target_std
            ),
            "Ensemble_Std_um": float(
                saved.loc[
                    cut_number, "ensemble_std"
                ]
            ),
            "Ensemble_Abs_Error_um": float(
                saved.loc[
                    cut_number, "absolute_error"
                ]
            )
        })

    del net

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


results = pd.DataFrame(rows)

assert len(results) == 9
assert results.notna().all().all()

output = WORK / "ig_cut_level_pilot.csv"
results.to_csv(output, index=False)

print("\nCUT-LEVEL IG PILOT RESULTS")

columns = [
    "Cutter",
    "Cut",
    "Full_IG_Cosine",
    "Channel_Cosine",
    "Channel_L1",
    "Prediction_Change_um",
    "Original_Completeness_um",
    "Perturbed_Completeness_um",
    "Ensemble_Std_um"
]

print(
    results[columns]
    .round(6)
    .to_string(index=False)
)

print("\nSaved: ig_cut_level_pilot.csv")


In [ ]:

import math
import numpy as np
import pandas as pd
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "V2CNNBiLSTM" in globals()
assert "cut_integrated_gradients" in globals(), (
    "Run the previous cut-level IG cell first."
)

with np.load(WORK / "phm_sequences.npz") as data:
    X_shap = data["X"].astype(np.float32)
    cutter_shap = data["cutter"].astype(int)
    cut_shap = data["cut"].astype(int)

AUDIT_CUTS = [1, 158, 315]
NOISE = 0.05
IG_STEPS = 128
N_CHANNELS = 7

# All 128 subsets of seven channels
subset_ids = np.arange(2 ** N_CHANNELS)

coalitions = np.array([
    [(subset >> j) & 1
     for j in range(N_CHANNELS)]
    for subset in subset_ids
], dtype=np.float32)

assert coalitions.shape == (128, 7)

# Precompute exact Shapley weights.
factorials = [
    math.factorial(k)
    for k in range(N_CHANNELS + 1)
]

shapley_weights = np.zeros(
    (128, N_CHANNELS), dtype=np.float64
)

for subset in range(128):
    size = int(coalitions[subset].sum())

    for channel in range(N_CHANNELS):
        if coalitions[subset, channel] == 0:
            shapley_weights[subset, channel] = (
                factorials[size] *
                factorials[N_CHANNELS - size - 1] /
                factorials[N_CHANNELS]
            )


def grouped_shap(net, x, target_std):
    """
    Exact grouped Shapley values.

    x: [4, 7, 2048] normalized sensor data.

    Missing channels are replaced by zeros,
    meaning the training-channel mean.

    Output: seven signed contributions in µm.
    """

    net.eval()
    values = np.empty(128, dtype=np.float64)

    # Process eight coalitions at once.
    with torch.no_grad():
        for start in range(0, 128, 8):
            masks = torch.from_numpy(
                coalitions[start:start + 8]
            ).to(DEVICE)

            # [8, 4, 7, 2048]
            samples = (
                x.unsqueeze(0) *
                masks[:, None, :, None]
            )

            batch = samples.reshape(
                -1, 7, 2048
            )

            predictions = net(batch).reshape(
                len(masks), 4
            ).mean(dim=1)

            values[start:start + 8] = (
                predictions.cpu().numpy()
            )

    shap_values = np.zeros(
        N_CHANNELS, dtype=np.float64
    )

    for channel in range(N_CHANNELS):
        for subset in range(128):

            if coalitions[subset, channel] == 1:
                continue

            with_channel = subset | (1 << channel)

            shap_values[channel] += (
                shapley_weights[subset, channel] *
                (values[with_channel] - values[subset])
            )

    shap_values *= target_std

    # Exact Shapley additivity check.
    delta_um = (
        values[127] - values[0]
    ) * target_std

    additivity_error = abs(
        shap_values.sum() - delta_um
    )

    return shap_values, additivity_error


def vector_cosine(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    denominator = (
        np.linalg.norm(a) *
        np.linalg.norm(b)
    )

    if denominator <= 1e-12:
        return np.nan

    return float(np.clip(
        np.dot(a, b) / denominator,
        -1, 1
    ))


def normalize_abs(a):
    importance = np.abs(
        np.asarray(a, dtype=np.float64)
    )

    return importance / max(
        importance.sum(), 1e-12
    )


rows = []
channel_rows = []

for held_out in [1, 4, 6]:

    print(
        f"\nIG VS EXACT GROUPED SHAP: C{held_out}",
        flush=True
    )

    checkpoint = torch.load(
        WORK / f"ensemble_C{held_out}_seed42.pt",
        map_location="cpu",
        weights_only=False
    )

    net = V2CNNBiLSTM().to(DEVICE)
    net.load_state_dict(
        checkpoint["model_state"]
    )
    net.eval()

    mean = np.asarray(
        checkpoint["channel_mean"],
        dtype=np.float32
    )

    std = np.asarray(
        checkpoint["channel_std"],
        dtype=np.float32
    )

    target_std = float(
        checkpoint["target_std"]
    )

    rng = np.random.default_rng(
        7000 + held_out
    )

    for cut_number in AUDIT_CUTS:

        indices = np.where(
            (cutter_shap == held_out) &
            (cut_shap == cut_number)
        )[0]

        assert len(indices) == 4

        raw = X_shap[indices]

        normalized = (raw - mean) / std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.transpose(0, 2, 1),
                dtype=np.float32
            )
        ).to(DEVICE)

        noise = rng.normal(
            0, NOISE, size=tuple(x.shape)
        ).astype(np.float32)

        perturbed = x + torch.from_numpy(
            noise
        ).to(DEVICE)

        # IG for the original cut.
        ig_original, ig_err1, _ = (
            cut_integrated_gradients(
                net, x, IG_STEPS
            )
        )

        # IG for the perturbed cut.
        ig_perturbed, ig_err2, _ = (
            cut_integrated_gradients(
                net, perturbed, IG_STEPS
            )
        )

        # Signed IG contributions per channel.
        ig_channel_original = (
            ig_original.sum(axis=(0, 2))
            * target_std
        )

        ig_channel_perturbed = (
            ig_perturbed.sum(axis=(0, 2))
            * target_std
        )

        # Exact grouped SHAP.
        shap_original, shap_err1 = (
            grouped_shap(net, x, target_std)
        )

        shap_perturbed, shap_err2 = (
            grouped_shap(
                net, perturbed, target_std
            )
        )

        ig_importance = normalize_abs(
            ig_channel_original
        )

        shap_importance = normalize_abs(
            shap_original
        )

        rows.append({
            "Cutter": f"C{held_out}",
            "Cut": cut_number,
            "IG_SHAP_Signed_Cosine": vector_cosine(
                ig_channel_original,
                shap_original
            ),
            "IG_SHAP_Importance_Cosine": vector_cosine(
                ig_importance,
                shap_importance
            ),
            "IG_Stability": vector_cosine(
                ig_channel_original,
                ig_channel_perturbed
            ),
            "SHAP_Stability": vector_cosine(
                shap_original,
                shap_perturbed
            ),
            "IG_Channel_L1": np.abs(
                normalize_abs(ig_channel_original) -
                normalize_abs(ig_channel_perturbed)
            ).sum(),
            "SHAP_Channel_L1": np.abs(
                normalize_abs(shap_original) -
                normalize_abs(shap_perturbed)
            ).sum(),
            "IG_Completeness_Error_um": (
                ig_err1 * target_std
            ),
            "SHAP_Additivity_Error_um": (
                shap_err1
            )
        })

        for j in range(N_CHANNELS):
            channel_rows.append({
                "Cutter": f"C{held_out}",
                "Cut": cut_number,
                "Channel": j + 1,
                "IG_Signed_um": (
                    ig_channel_original[j]
                ),
                "SHAP_Signed_um": (
                    shap_original[j]
                )
            })

        print(
            f"Cut {cut_number} completed",
            flush=True
        )

    del net
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


comparison = pd.DataFrame(rows)
channels = pd.DataFrame(channel_rows)

assert len(comparison) == 9
assert len(channels) == 63
assert comparison.notna().all().all()

comparison.to_csv(
    WORK / "ig_vs_grouped_shap_pilot.csv",
    index=False
)

channels.to_csv(
    WORK / "ig_vs_grouped_shap_channels.csv",
    index=False
)

print("\nIG VS GROUPED SHAP PILOT RESULTS")

print(
    comparison.round(6).to_string(index=False)
)

print("\nMEAN STABILITY BY METHOD")

print(
    comparison.groupby("Cutter")[
        ["IG_Stability", "SHAP_Stability",
         "IG_Channel_L1", "SHAP_Channel_L1"]
    ].mean().round(6).to_string()
)

print("\nSaved:")
print("ig_vs_grouped_shap_pilot.csv")
print("ig_vs_grouped_shap_channels.csv")


In [ ]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import spearmanr

WORK = Path("/kaggle/working")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

for function_name in [
    "V2CNNBiLSTM",
    "cut_integrated_gradients",
    "grouped_shap"
]:
    assert function_name in globals(), (
        f"Run the previous experiment cell "
        f"to define {function_name}."
    )

with np.load(WORK / "phm_sequences.npz") as data:
    X_main = data["X"].astype(np.float32)
    cutter_main = data["cutter"].astype(int)
    cut_main = data["cut"].astype(int)

SELECTED_CUTS = np.linspace(
    1, 315, 15, dtype=int
)

REPETITIONS = 2
NOISE_LEVEL = 0.05
IG_STEPS = 128

def cosine_v3(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator < 1e-12:
        return np.nan

    return float(np.clip(
        np.dot(a, b) / denominator, -1, 1
    ))

def normalized_importance_v3(a):
    values = np.abs(np.asarray(a, dtype=np.float64))
    return values / max(values.sum(), 1e-12)

records = []

for held_out in [1, 4, 6]:
    print(
        f"\nEXPLANATION STABILITY STUDY: C{held_out}",
        flush=True
    )

    checkpoint = torch.load(
        WORK / f"ensemble_C{held_out}_seed42.pt",
        map_location="cpu",
        weights_only=False
    )

    model = V2CNNBiLSTM().to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    channel_mean = np.asarray(
        checkpoint["channel_mean"], dtype=np.float32
    )
    channel_std = np.asarray(
        checkpoint["channel_std"], dtype=np.float32
    )
    target_std = float(checkpoint["target_std"])

    ensemble = pd.read_csv(
        WORK / f"ensemble_C{held_out}_test_combined.csv"
    ).set_index("cut")

    rng = np.random.default_rng(12000 + held_out)

    for cut_number in SELECTED_CUTS:
        indices = np.where(
            (cutter_main == held_out) &
            (cut_main == cut_number)
        )[0]

        assert len(indices) == 4

        normalized = (
            X_main[indices] - channel_mean
        ) / channel_std

        x = torch.from_numpy(
            np.ascontiguousarray(
                normalized.transpose(0, 2, 1),
                dtype=np.float32
            )
        ).to(DEVICE)

        # Explain original four-window cut prediction once.
        ig_original, ig_error, _ = (
            cut_integrated_gradients(
                model, x, IG_STEPS
            )
        )

        ig_original_channels = (
            ig_original.sum(axis=(0, 2)) * target_std
        )

        shap_original, shap_error = (
            grouped_shap(model, x, target_std)
        )

        original_ig_importance = (
            normalized_importance_v3(
                ig_original_channels
            )
        )

        original_shap_importance = (
            normalized_importance_v3(
                shap_original
            )
        )

        ref = ensemble.loc[cut_number]

        for repetition in range(1, REPETITIONS + 1):
            noise = rng.normal(
                0,
                NOISE_LEVEL,
                size=tuple(x.shape)
            ).astype(np.float32)

            x_perturbed = x + torch.from_numpy(
                noise
            ).to(DEVICE)

            ig_perturbed, perturbed_ig_error, _ = (
                cut_integrated_gradients(
                    model, x_perturbed, IG_STEPS
                )
            )

            ig_perturbed_channels = (
                ig_perturbed.sum(axis=(0, 2)) *
                target_std
            )

            shap_perturbed, perturbed_shap_error = (
                grouped_shap(
                    model, x_perturbed, target_std
                )
            )

            perturbed_ig_importance = (
                normalized_importance_v3(
                    ig_perturbed_channels
                )
            )

            perturbed_shap_importance = (
                normalized_importance_v3(
                    shap_perturbed
                )
            )

            with torch.no_grad():
                prediction_change = abs(
                    model(x).mean().item() -
                    model(x_perturbed).mean().item()
                ) * target_std

            records.append({
                "Cutter": f"C{held_out}",
                "Cut": int(cut_number),
                "Repetition": repetition,
                "Ensemble_Std_um": float(
                    ref["ensemble_std"]
                ),
                "Ensemble_Abs_Error_um": float(
                    ref["absolute_error"]
                ),
                "IG_SHAP_Agreement": cosine_v3(
                    ig_original_channels,
                    shap_original
                ),
                "IG_Full_Instability": (
                    1 - cosine_v3(
                        ig_original, ig_perturbed
                    )
                ),
                "IG_Channel_Instability": (
                    1 - cosine_v3(
                        ig_original_channels,
                        ig_perturbed_channels
                    )
                ),
                "SHAP_Channel_Instability": (
                    1 - cosine_v3(
                        shap_original,
                        shap_perturbed
                    )
                ),
                "IG_Channel_L1": float(
                    np.abs(
                        original_ig_importance -
                        perturbed_ig_importance
                    ).sum()
                ),
                "SHAP_Channel_L1": float(
                    np.abs(
                        original_shap_importance -
                        perturbed_shap_importance
                    ).sum()
                ),
                "Prediction_Change_um": float(
                    prediction_change
                ),
                "IG_Error_um": float(
                    ig_error * target_std
                ),
                "IG_Perturbed_Error_um": float(
                    perturbed_ig_error * target_std
                ),
                "SHAP_Error_um": float(
                    shap_error
                ),
                "SHAP_Perturbed_Error_um": float(
                    perturbed_shap_error
                )
            })

        print(f"Cut {cut_number} completed", flush=True)

    # Save completed cutter immediately.
    partial = pd.DataFrame(records)
    partial.to_csv(
        WORK / "explanation_stability_v3_progress.csv",
        index=False
    )

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ==========================================
# SUMMARY
# ==========================================

results = pd.DataFrame(records)

assert len(results) == 90
assert results.notna().all().all()

results.to_csv(
    WORK / "explanation_stability_v3.csv",
    index=False
)

# Average the two noise repetitions per cut.
# Treat the cut, not the noise repetition,
# as the observational unit.
per_cut = (
    results.groupby(
        ["Cutter", "Cut"], as_index=False
    ).agg({
        "Ensemble_Std_um": "first",
        "Ensemble_Abs_Error_um": "first",
        "IG_SHAP_Agreement": "first",
        "IG_Full_Instability": "mean",
        "IG_Channel_Instability": "mean",
        "SHAP_Channel_Instability": "mean",
        "IG_Channel_L1": "mean",
        "SHAP_Channel_L1": "mean",
        "Prediction_Change_um": "mean"
    })
)

assert len(per_cut) == 45

per_cut.to_csv(
    WORK / "explanation_stability_v3_per_cut.csv",
    index=False
)

print("\nEXPLANATION STABILITY V3 SUMMARY")

print(
    per_cut.groupby("Cutter")[
        [
            "IG_SHAP_Agreement",
            "IG_Full_Instability",
            "IG_Channel_L1",
            "SHAP_Channel_L1"
        ]
    ].mean().round(6).to_string()
)

print("\nUNCERTAINTY VS EXPLANATION INSTABILITY")

for cutter in ["C1", "C4", "C6"]:
    part = per_cut[per_cut["Cutter"] == cutter]

    for metric in [
        "IG_Full_Instability",
        "IG_Channel_L1",
        "SHAP_Channel_L1"
    ]:
        rho, _ = spearmanr(
            part["Ensemble_Std_um"],
            part[metric]
        )

        print(
            cutter,
            "|",
            metric,
            "| rho =",
            round(float(rho), 3)
        )

print("\nSaved:")
print("explanation_stability_v3.csv")
print("explanation_stability_v3_per_cut.csv")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr, rankdata

WORK = Path("/kaggle/working")

PATH = WORK / "explanation_stability_v3_per_cut.csv"
assert PATH.exists(), "Missing V3 per-cut results."

df = pd.read_csv(PATH)

assert len(df) == 45
assert df.groupby("Cutter").size().eq(15).all()

METRICS = [
    "IG_Full_Instability",
    "IG_Channel_L1",
    "SHAP_Channel_L1"
]

REQUIRED = [
    "Cutter",
    "Cut",
    "Ensemble_Std_um",
    "Ensemble_Abs_Error_um"
] + METRICS

assert df[REQUIRED].notna().all().all()

# Label tool-life stages.
df["Stage"] = pd.cut(
    df["Cut"],
    bins=[0, 105, 210, 315],
    labels=["Early", "Middle", "Late"]
)

def safe_spearman(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if (
        len(a) < 3
        or np.std(a) < 1e-12
        or np.std(b) < 1e-12
    ):
        return np.nan

    return float(spearmanr(a, b).statistic)

def partial_rank_correlation(x, y, control):
    """
    Exploratory partial rank correlation:
    regress ranked x and ranked y separately
    on the ranked control, then correlate
    the residuals.

    This is descriptive, not a causal test.
    """
    x = rankdata(np.asarray(x, dtype=float))
    y = rankdata(np.asarray(y, dtype=float))
    z = rankdata(np.asarray(control, dtype=float))

    design = np.column_stack([
        np.ones(len(z)),
        z
    ])

    beta_x = np.linalg.lstsq(
        design, x, rcond=None
    )[0]

    beta_y = np.linalg.lstsq(
        design, y, rcond=None
    )[0]

    residual_x = x - design @ beta_x
    residual_y = y - design @ beta_y

    if (
        np.std(residual_x) < 1e-12
        or np.std(residual_y) < 1e-12
    ):
        return np.nan

    return float(np.corrcoef(
        residual_x, residual_y
    )[0, 1])

rows = []

for cutter in ["C1", "C4", "C6"]:
    part = df[df["Cutter"] == cutter].sort_values("Cut")

    uncertainty = part["Ensemble_Std_um"]
    cut_number = part["Cut"]

    for metric in METRICS:
        rows.append({
            "Cutter": cutter,
            "Metric": metric,
            "Uncertainty_vs_Instability": safe_spearman(
                uncertainty, part[metric]
            ),
            "Cut_vs_Uncertainty": safe_spearman(
                cut_number, uncertainty
            ),
            "Cut_vs_Instability": safe_spearman(
                cut_number, part[metric]
            ),
            "Error_vs_Instability": safe_spearman(
                part["Ensemble_Abs_Error_um"],
                part[metric]
            ),
            "Partial_Rho_Controlling_Cut": (
                partial_rank_correlation(
                    uncertainty,
                    part[metric],
                    cut_number
                )
            )
        })

diagnostics = pd.DataFrame(rows)

print("\nUNCERTAINTY AND EXPLANATION CONFOUNDING CHECK")
print(diagnostics.round(3).to_string(index=False))

print("\nSTABILITY BY TOOL-LIFE STAGE")

stage_summary = (
    df.groupby(
        ["Cutter", "Stage"],
        observed=True
    )
    .agg(
        N_Cuts=("Cut", "count"),
        Mean_Uncertainty=(
            "Ensemble_Std_um", "mean"
        ),
        Mean_Abs_Error=(
            "Ensemble_Abs_Error_um", "mean"
        ),
        Mean_IG_Full_Instability=(
            "IG_Full_Instability", "mean"
        ),
        Mean_IG_Channel_L1=(
            "IG_Channel_L1", "mean"
        ),
        Mean_SHAP_Channel_L1=(
            "SHAP_Channel_L1", "mean"
        )
    )
    .reset_index()
)

print(stage_summary.round(5).to_string(index=False))

diagnostics.to_csv(
    WORK / "explanation_confounding_diagnostics.csv",
    index=False
)

stage_summary.to_csv(
    WORK / "explanation_stability_by_stage.csv",
    index=False
)

print("\nSaved:")
print("explanation_confounding_diagnostics.csv")
print("explanation_stability_by_stage.csv")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

THRESHOLDS = [150.0, 180.0, 200.0]

# Hypothetical missed-event costs relative
# to the cost of one false alarm.
COST_SCENARIOS = [1.0, 5.0, 20.0]

POLICIES = {
    "Random Forest": "rf_prediction",
    "Ensemble Mean": "ensemble_mean",
    "Standard Upper": "standard_upper",
    "Adaptive Upper": "adaptive_upper"
}

rows = []
decisions = []

for cutter in [1, 4, 6]:

    rf_path = WORK / f"rf_v2_C{cutter}_predictions.csv"
    cp_path = WORK / (
        f"ensemble_C{cutter}_conformal_comparison.csv"
    )

    assert rf_path.exists(), f"Missing {rf_path.name}"
    assert cp_path.exists(), f"Missing {cp_path.name}"

    rf = pd.read_csv(rf_path)
    cp = pd.read_csv(cp_path)

    assert len(rf) == 315
    assert len(cp) == 315

    rf = rf.rename(columns={
        "predicted": "rf_prediction",
        "actual": "rf_actual"
    })

    combined = cp.merge(
        rf[["cutter", "cut", "rf_actual",
            "rf_prediction"]],
        on=["cutter", "cut"],
        how="inner",
        validate="one_to_one"
    )

    assert len(combined) == 315
    assert combined["cut"].is_unique
    assert set(combined["cut"]) == set(range(1, 316))

    assert np.allclose(
        combined["actual"],
        combined["rf_actual"],
        atol=1e-3,
        rtol=0
    ), "Target values do not match."

    required = [
        "actual",
        "rf_prediction",
        "ensemble_mean",
        "standard_upper",
        "adaptive_upper"
    ]

    assert np.isfinite(
        combined[required].to_numpy(dtype=float)
    ).all()

    for threshold in THRESHOLDS:

        # Ground truth based on measured wear.
        actual_high = (
            combined["actual"].to_numpy()
            >= threshold
        )

        for policy, column in POLICIES.items():

            # Warning based only on model outputs.
            warning = (
                combined[column].to_numpy()
                >= threshold
            )

            tp = int(np.sum(warning & actual_high))
            fp = int(np.sum(warning & ~actual_high))
            fn = int(np.sum(~warning & actual_high))
            tn = int(np.sum(~warning & ~actual_high))

            assert tp + fp + fn + tn == 315

            recall = (
                tp / (tp + fn)
                if tp + fn > 0 else np.nan
            )

            precision = (
                tp / (tp + fp)
                if tp + fp > 0 else np.nan
            )

            false_alarm_rate = (
                fp / (fp + tn)
                if fp + tn > 0 else np.nan
            )

            for missed_cost in COST_SCENARIOS:

                # Illustrative loss, NOT actual
                # manufacturing costs.
                total_cost = (
                    fp * 1.0 + fn * missed_cost
                )

                rows.append({
                    "Cutter": f"C{cutter}",
                    "Threshold_um": threshold,
                    "Policy": policy,
                    "Missed_Event_Cost": missed_cost,
                    "Actual_High_Cuts": int(
                        actual_high.sum()
                    ),
                    "Warnings": int(warning.sum()),
                    "TP": tp,
                    "FP": fp,
                    "FN": fn,
                    "TN": tn,
                    "Recall": recall,
                    "Precision": precision,
                    "False_Alarm_Rate": false_alarm_rate,
                    "Total_Scenario_Cost": total_cost,
                    "Cost_Per_Cut": total_cost / 315
                })

            decisions.extend([
                {
                    "Cutter": f"C{cutter}",
                    "Cut": int(cut),
                    "Threshold_um": threshold,
                    "Policy": policy,
                    "Actual_High_Wear": bool(high),
                    "Warning": bool(flag)
                }
                for cut, high, flag in zip(
                    combined["cut"].to_numpy(),
                    actual_high,
                    warning
                )
            ])

metrics = pd.DataFrame(rows)
decision_df = pd.DataFrame(decisions)

metrics.to_csv(
    WORK / "replacement_policy_scenarios.csv",
    index=False
)

decision_df.to_csv(
    WORK / "replacement_policy_decisions.csv",
    index=False
)

# Display a manageable initial scenario.
print("\nREPLACEMENT POLICY RESULTS")
print("Threshold: 180 µm")
print("False-alarm cost: 1")
print("Missed high-wear cost: 5")

selected = metrics[
    (metrics["Threshold_um"] == 180.0) &
    (metrics["Missed_Event_Cost"] == 5.0)
].copy()

print(
    selected[
        [
            "Cutter",
            "Policy",
            "Actual_High_Cuts",
            "Warnings",
            "TP",
            "FP",
            "FN",
            "Recall",
            "Cost_Per_Cut"
        ]
    ].round(3).to_string(index=False)
)

print("\nTOTAL COST BY POLICY")
print(
    selected.groupby("Policy")[
        "Total_Scenario_Cost"
    ].sum().sort_values().round(2).to_string()
)

print("\nSENSITIVITY: TOTAL COSTS ACROSS SCENARIOS")

scenario = (
    metrics.groupby(
        ["Threshold_um", "Missed_Event_Cost", "Policy"]
    )["Total_Scenario_Cost"]
    .sum()
    .reset_index()
)

print(
    scenario.pivot_table(
        index=["Threshold_um", "Missed_Event_Cost"],
        columns="Policy",
        values="Total_Scenario_Cost"
    ).round(2).to_string()
)

print("\nSaved:")
print("replacement_policy_scenarios.csv")
print("replacement_policy_decisions.csv")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

WORK = Path("/kaggle/working")
path = WORK / "replacement_policy_decisions.csv"

assert path.exists(), "Replacement decisions file missing."

df = pd.read_csv(path)

# Normalize Boolean fields safely.
for col in ["Actual_High_Wear", "Warning"]:
    df[col] = (
        df[col].astype(str).str.lower()
        .map({"true": True, "false": False})
    )
    assert df[col].notna().all(), f"Invalid values in {col}"

rows = []

for (cutter, threshold, policy), part in df.groupby(
    ["Cutter", "Threshold_um", "Policy"]
):
    part = part.sort_values("Cut")

    assert len(part) == 315
    assert part["Cut"].is_unique
    assert set(part["Cut"]) == set(range(1, 316))

    actual_crossings = part.loc[
        part["Actual_High_Wear"], "Cut"
    ]

    warnings = part.loc[
        part["Warning"], "Cut"
    ]

    first_actual = (
        int(actual_crossings.iloc[0])
        if len(actual_crossings) else None
    )

    first_warning = (
        int(warnings.iloc[0])
        if len(warnings) else None
    )

    if first_actual is None:
        status = (
            "Unnecessary warning"
            if first_warning is not None
            else "Correctly no warning"
        )
        lead = np.nan

    elif first_warning is None:
        status = "No warning despite high wear"
        lead = np.nan

    else:
        lead = first_actual - first_warning

        if lead > 0:
            status = "Early warning"
        elif lead == 0:
            status = "At threshold crossing"
        else:
            status = "Late warning"

    rows.append({
        "Cutter": cutter,
        "Threshold_um": threshold,
        "Policy": policy,
        "First_Actual_High_Cut": first_actual,
        "First_Warning_Cut": first_warning,
        "Lead_Cuts": lead,
        "Status": status
    })

results = pd.DataFrame(rows)

assert len(results) == 36

results.to_csv(
    WORK / "replacement_first_warning_analysis.csv",
    index=False
)

print("\nFIRST-WARNING ANALYSIS: 180 UM")

selected = results[
    results["Threshold_um"] == 180.0
]

print(
    selected[
        [
            "Cutter",
            "Policy",
            "First_Actual_High_Cut",
            "First_Warning_Cut",
            "Lead_Cuts",
            "Status"
        ]
    ].to_string(index=False)
)

print("\nFIRST-WARNING STATUS COUNTS")

print(
    results.groupby(
        ["Threshold_um", "Policy", "Status"]
    ).size().to_string()
)

print("\nSaved: replacement_first_warning_analysis.csv")


In [ ]:

from pathlib import Path
import zipfile

WORK = Path("/kaggle/working")
BACKUP = WORK / "PHM2010_Research_Backup.zip"

files = [
    p for p in WORK.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".csv", ".pt", ".npz"}
]

assert len(files) > 0, "No research files found."

with zipfile.ZipFile(
    BACKUP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as archive:
    for file in sorted(files):
        archive.write(file, arcname=file.name)

with zipfile.ZipFile(BACKUP, "r") as archive:
    assert archive.testzip() is None
    print("Verified files:", len(archive.namelist()))

print("BACKUP CREATED")
print("File:", BACKUP.name)
print("Size:", round(BACKUP.stat().st_size / 1024**2, 2), "MB")

print("\nIMPORTANT FILES")
for name in [
    "phm_sequences.npz",
    "phm_protocol_v2_splits.csv",
    "rf_protocol_v2_results.csv",
    "cnn_protocol_v2_results.csv",
    "ensemble_evaluation_results.csv",
    "ensemble_conformal_comparison.csv",
    "explanation_stability_v3_per_cut.csv",
    "replacement_policy_scenarios.csv",
    "replacement_first_warning_analysis.csv"
]:
    print(
        name,
        "FOUND" if (WORK / name).exists()
        else "MISSING"
    )


In [ ]:
"""PHM2010 paper asset generator; run in Kaggle after the research notebook.
Reads saved CSVs from /kaggle/working (or --workdir), writes PNG/PDF/CSV and ZIP.
No model training; no invented results.
"""
from pathlib import Path
import argparse
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

parser = argparse.ArgumentParser()
parser.add_argument('--workdir', default='/kaggle/working')
args, _ = parser.parse_known_args()
WORK = Path(args.workdir)
BASE = WORK / 'PHM2010_Paper_Assets'
FIG = BASE / 'figures'
TAB = BASE / 'tables'
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,
                     'axes.labelsize':10,'legend.fontsize':9,'figure.dpi':120,
                     'savefig.dpi':300,'axes.spines.top':False,'axes.spines.right':False})
made_figs, made_tables, missing = [], [], []

def read(name, required=None):
    p=WORK/name
    if not p.exists():
        missing.append(name)
        print('SKIP (missing):', name)
        return None
    d=pd.read_csv(p)
    if required:
        absent=sorted(set(required)-set(d.columns))
        if absent:
            print('SKIP (columns missing):',name,absent)
            return None
    return d

def table(df, name):
    out=TAB/name
    df.to_csv(out,index=False)
    made_tables.append(str(out.relative_to(BASE)))
    print('TABLE:', out.name)

def save(name, fig):
    fig.tight_layout()
    png=FIG/(name+'.png'); pdf=FIG/(name+'.pdf')
    fig.savefig(png,dpi=300,bbox_inches='tight',facecolor='white')
    fig.savefig(pdf,bbox_inches='tight',facecolor='white')
    plt.close(fig)
    made_figs.append(str(png.relative_to(BASE)))
    print('FIGURE:',png.name)

def panel_letters(axs):
    for i,ax in enumerate(np.array(axs).ravel()):
        ax.text(-.12,1.04,chr(65+i),transform=ax.transAxes,weight='bold',fontsize=12)

def colors_for_models(models):
    cmap=plt.get_cmap('tab10')
    predefined={'Random Forest V2':cmap(0),'CNN-BiLSTM V2':cmap(1),'Five-model ensemble':cmap(2)}
    return [predefined.get(m,cmap(i%10)) for i,m in enumerate(models)]

# Table 1 + Figure 1: predictive accuracy
rf=read('rf_protocol_v2_results.csv',['Test_Cutter','MAE','RMSE','R2'])
cnn=read('cnn_protocol_v2_results.csv',['Cutter','MAE','RMSE','R2'])
ens=read('ensemble_evaluation_results.csv',['Cutter','MAE','RMSE','R2','Mean_Std','Std_Error_Rho'])
if all(d is not None for d in (rf,cnn,ens)):
    parts=[]
    for title,d,key in [('Random Forest V2',rf,'Test_Cutter'),('CNN-BiLSTM V2',cnn,'Cutter'),('Five-model ensemble',ens,'Cutter')]:
        t=d[[key,'MAE','RMSE','R2']].rename(columns={key:'Cutter'}).copy();t.insert(0,'Model',title)
        parts.append(t)
    acc=pd.concat(parts,ignore_index=True)
    macro=acc.groupby('Model',as_index=False)[['MAE','RMSE','R2']].mean()
    table(acc,'T01_prediction_accuracy_by_cutter.csv')
    table(macro,'T02_prediction_accuracy_macro.csv')
    names=['Random Forest V2','CNN-BiLSTM V2','Five-model ensemble']
    fig,axs=plt.subplots(1,3,figsize=(13.5,4.1))
    for ax,metric,ylabel in zip(axs,['MAE','RMSE','R2'],['MAE (µm), lower better','RMSE (µm), lower better','R², higher better']):
        x=np.arange(3);width=.25
        for i,name in enumerate(names):
            d=acc[acc.Model==name].set_index('Cutter').reindex(['C1','C4','C6'])
            ax.bar(x+(i-1)*width,d[metric],width,label=name,color=colors_for_models(names)[i])
        ax.set_xticks(x,['C1','C4','C6']);ax.set_ylabel(ylabel);ax.set_title(metric+' by held-out cutter')
        if metric=='R2': ax.axhline(0,color='gray',lw=.8)
    axs[0].legend(loc='upper left',fontsize=8)
    panel_letters(axs)
    save('F01_model_accuracy_loco',fig)

# Figure 2: progression actual vs prediction and ensemble uncertainty
preds=[]
for c in [1,4,6]:
    r=read(f'rf_v2_C{c}_predictions.csv',['cut','actual','predicted'])
    e=read(f'ensemble_C{c}_test_combined.csv',['cut','actual','ensemble_mean','ensemble_std'])
    if r is not None and e is not None:
        joined=e.merge(r[['cut','predicted','actual']].rename(columns={'predicted':'RF_predicted','actual':'RF_actual'}),on='cut',validate='one_to_one')
        assert np.allclose(joined.actual,joined.RF_actual,atol=1e-2)
        joined.insert(0,'Cutter',f'C{c}')
        preds.append(joined)
if len(preds)==3:
    allpred=pd.concat(preds,ignore_index=True)
    table(allpred[['Cutter','cut','actual','RF_predicted','ensemble_mean','ensemble_std']],'T03_predictions_by_cut.csv')
    fig,axs=plt.subplots(3,1,figsize=(11,9),sharex=True)
    for ax,c in zip(axs,[1,4,6]):
        d=allpred[allpred.Cutter==f'C{c}'].sort_values('cut')
        x=d.cut.to_numpy(); y=d.ensemble_mean.to_numpy(); sd=d.ensemble_std.to_numpy()
        ax.plot(x,d.actual,color='black',lw=1.8,label='Measured wear')
        ax.plot(x,d.RF_predicted,lw=1.1,label='Random Forest')
        ax.plot(x,y,lw=1.1,label='Ensemble mean')
        ax.fill_between(x,y-sd,y+sd,alpha=.16,label='Ensemble ±1 SD (not calibrated)')
        ax.set_title(f'C{c}: unseen cutter');ax.set_ylabel('Wear (µm)')
    axs[0].legend(ncol=4,fontsize=8,loc='upper left');axs[-1].set_xlabel('Cut number')
    save('F02_wear_trajectories',fig)

# Figure 3: ensemble disagreement vs absolute error
if len(preds)==3:
    fig,axs=plt.subplots(1,3,figsize=(13,3.8))
    for ax,c in zip(axs,[1,4,6]):
        d=allpred[allpred.Cutter==f'C{c}']
        ax.scatter(d.ensemble_std,abs(d.actual-d.ensemble_mean),s=12,alpha=.65)
        ax.set_title(f'C{c}');ax.set_xlabel('Ensemble SD (µm)');ax.set_ylabel('Absolute prediction error (µm)')
    panel_letters(axs);save('F03_uncertainty_vs_error',fig)
    table(ens[['Cutter','MAE','RMSE','R2','Mean_Std','Std_Error_Rho']],'T04_ensemble_uncertainty_statistics.csv')

# Figure 4/5: conformal coverage and widths, interval scores
cp=read('ensemble_conformal_comparison.csv',['Cutter','Method','Coverage_pct','Mean_Width_um','Interval_Score'])
if cp is not None:
    table(cp,'T05_conformal_coverage_width_score.csv')
    fig,axs=plt.subplots(1,3,figsize=(13.4,4))
    for ax,(metric,ylabel) in zip(axs,[('Coverage_pct','Empirical coverage (%)'),('Mean_Width_um','Mean interval width (µm)'),('Interval_Score','Interval score (lower better)')]):
        x=np.arange(3);w=.34
        for i,method in enumerate(['Standard','Adaptive']):
            values=cp[cp.Method==method].set_index('Cutter').reindex(['C1','C4','C6'])[metric].to_numpy()
            ax.bar(x+(i-.5)*w,values,width=w,label=method)
        ax.set_xticks(x,['C1','C4','C6']);ax.set_ylabel(ylabel);ax.set_title(ylabel)
        if metric=='Coverage_pct': ax.axhline(90,color='black',ls='--',lw=1,label='Nominal 90%')
    axs[0].legend(fontsize=8);panel_letters(axs);save('F04_conformal_method_comparison',fig)

# Figure 5: actual vs intervals to convey calibration failure (sampled every 10th cut for readability)
intervals=[]
for c in [1,4,6]:
    d=read(f'ensemble_C{c}_conformal_comparison.csv', ['cut','actual','ensemble_mean','standard_lower','standard_upper','adaptive_lower','adaptive_upper'])
    if d is not None:
        d.insert(0,'Cutter',f'C{c}');intervals.append(d)
if len(intervals)==3:
    fig,axs=plt.subplots(3,1,figsize=(11,9),sharex=True)
    for ax,c in zip(axs,[1,4,6]):
        d=intervals[[1,4,6].index(c)].sort_values('cut')
        # 315 points, draw all lines but light interval boundaries.
        ax.plot(d.cut,d.actual,color='black',lw=1.5,label='Measured')
        ax.plot(d.cut,d.ensemble_mean,color='tab:blue',lw=1,label='Ensemble mean')
        ax.fill_between(d.cut,d.adaptive_lower,d.adaptive_upper,alpha=.15,color='tab:orange',label='Adaptive interval')
        ax.plot(d.cut,d.standard_upper,color='tab:green',ls='--',lw=.75,label='Standard upper')
        ax.set_title(f'C{c} — nominal 90% intervals, empirical coverage may differ')
        ax.set_ylabel('Wear (µm)')
    axs[0].legend(ncol=4,fontsize=8);axs[-1].set_xlabel('Cut number')
    save('F05_conformal_intervals_trajectories',fig)

# Fig 6: test stage coverage
stage=read('conformal_stage_analysis.csv',['Cutter','Stage','Coverage_pct','Above_upper_pct','Below_lower_pct'])
if stage is not None:
    table(stage,'T06_single_cnn_conformal_stage_diagnostics.csv')
    fig,axs=plt.subplots(1,3,figsize=(12.6,3.9),sharey=True)
    for ax,c in zip(axs,['C1','C4','C6']):
        d=stage[stage.Cutter==c].set_index('Stage').reindex(['Early','Middle','Late'])
        ax.bar(np.arange(3),d.Coverage_pct)
        ax.set_title(c);ax.set_xticks(range(3),['Early','Middle','Late'])
        ax.axhline(90,color='black',ls='--',lw=1)
        ax.set_ylim(0,105)
    axs[0].set_ylabel('Empirical coverage (%)');panel_letters(axs)
    save('F06_single_cnn_coverage_by_stage',fig)

# Fig 7: IG / SHAP uncertainty association
exp=read('explanation_stability_v3_per_cut.csv',['Cutter','Cut','Ensemble_Std_um','IG_Full_Instability','IG_Channel_L1','SHAP_Channel_L1','IG_SHAP_Agreement'])
if exp is not None:
    table(exp,'T07_explanation_stability_per_cut.csv')
    metrics=[('IG_Full_Instability','Full-resolution IG instability'),('IG_Channel_L1','IG channel L1 change'),('SHAP_Channel_L1','SHAP channel L1 change')]
    fig,axs=plt.subplots(3,3,figsize=(12,10))
    for i,c in enumerate(['C1','C4','C6']):
        d=exp[exp.Cutter==c]
        for j,(metric,label) in enumerate(metrics):
            ax=axs[i,j]
            ax.scatter(d.Ensemble_Std_um,d[metric],s=28,alpha=.8)
            rho=d[['Ensemble_Std_um',metric]].corr(method='spearman').iloc[0,1]
            ax.set_title(f'{c}: ρ={rho:.2f}')
            if i==2:ax.set_xlabel('Ensemble SD (µm)')
            if j==0:ax.set_ylabel(label)
            else: ax.set_ylabel(label)
    save('F07_uncertainty_vs_explanation_instability',fig)
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    for ax,(metric,label) in zip(axs,[('IG_SHAP_Agreement','IG–SHAP signed cosine'),('IG_Full_Instability','Full-resolution IG instability')]):
        for c in ['C1','C4','C6']:
            d=exp[exp.Cutter==c].sort_values('Cut')
            ax.plot(d.Cut,d[metric],marker='.',label=c)
        ax.set_xlabel('Cut number');ax.set_ylabel(label)
    axs[0].legend();save('F08_explanation_agreement_over_tool_life',fig)

# Table 8 + figure 9: confounding and stage
conf=read('explanation_confounding_diagnostics.csv')
if conf is not None:table(conf,'T08_uncertainty_explanation_confounding.csv')
expstage=read('explanation_stability_by_stage.csv',['Cutter','Stage','Mean_IG_Full_Instability','Mean_IG_Channel_L1','Mean_SHAP_Channel_L1'])
if expstage is not None:
    table(expstage,'T09_explanation_stability_by_stage.csv')
    fig,ax=plt.subplots(figsize=(7.3,4.5))
    for c in ['C1','C4','C6']:
        d=expstage[expstage.Cutter==c].set_index('Stage').reindex(['Early','Middle','Late'])
        ax.plot(['Early','Middle','Late'],d.Mean_IG_Full_Instability,marker='o',label=c)
    ax.set_ylabel('Mean full-resolution IG instability');ax.set_xlabel('Tool-life stage');ax.legend()
    save('F09_ig_stability_by_stage',fig)

# Fig 10: per-cut operational confusion matrix at 180 um
policy=read('replacement_policy_scenarios.csv',['Cutter','Threshold_um','Policy','Missed_Event_Cost','TP','FP','FN','TN','Total_Scenario_Cost'])
if policy is not None:
    table(policy,'T10_warning_policy_all_scenarios.csv')
    subset=policy[(policy.Threshold_um==180)&(policy.Missed_Event_Cost==5)].copy()
    if len(subset)==12:
        pivot=subset.pivot(index='Policy',columns='Cutter',values='FN').reindex(['Random Forest','Ensemble Mean','Standard Upper','Adaptive Upper'])
        fig,axs=plt.subplots(1,2,figsize=(12,4.5))
        for ax,column,ttl in [(axs[0],'FN','Missed high-wear cuts'),(axs[1],'FP','False alarms')]:
            p=subset.pivot(index='Policy',columns='Cutter',values=column).reindex(['Random Forest','Ensemble Mean','Standard Upper','Adaptive Upper'])[['C1','C4','C6']]
            im=ax.imshow(p.to_numpy(),aspect='auto',cmap='Blues')
            ax.set_yticks(range(4),p.index);ax.set_xticks(range(3),p.columns);ax.set_title(ttl+' (180 µm)')
            for i in range(4):
                for j in range(3):
                    ax.text(j,i,str(int(p.iloc[i,j])),ha='center',va='center',color='black')
        save('F10_warning_false_alarms_and_misses',fig)
    # cost sensitivity: each threshold, various missed event cost ratios
    scenario=policy.groupby(['Threshold_um','Missed_Event_Cost','Policy'],as_index=False).Total_Scenario_Cost.sum()
    fig,axs=plt.subplots(1,3,figsize=(13.5,4),sharey=True)
    for ax,thr in zip(axs,[150,180,200]):
        part=scenario[scenario.Threshold_um==thr]
        for name in ['Random Forest','Ensemble Mean','Standard Upper','Adaptive Upper']:
            d=part[part.Policy==name].sort_values('Missed_Event_Cost')
            ax.plot(d.Missed_Event_Cost,d.Total_Scenario_Cost,marker='o',label=name)
        ax.set_title(f'Threshold {thr} µm');ax.set_xlabel('Relative cost of missed event')
        ax.set_xticks([1,5,20])
    axs[0].set_ylabel('Total illustrative cost (945 cut-level opportunities)')
    axs[0].legend(fontsize=8);save('F11_warning_cost_sensitivity',fig)

# Fig 12: first warning timing, do not represent as actual replacement
first=read('replacement_first_warning_analysis.csv',['Cutter','Threshold_um','Policy','First_Actual_High_Cut','First_Warning_Cut','Lead_Cuts','Status'])
if first is not None:
    table(first,'T11_first_warning_timing.csv')
    d=first[(first.Threshold_um==180)&(first.Cutter.isin(['C4','C6']))]
    fig,axs=plt.subplots(2,1,figsize=(10,6),sharex=True)
    for ax,c in zip(axs,['C4','C6']):
        p=d[d.Cutter==c].set_index('Policy')
        names=['Random Forest','Ensemble Mean','Standard Upper','Adaptive Upper']
        for i,name in enumerate(names):
            val=p.loc[name,'First_Warning_Cut']
            if pd.notna(val):ax.scatter(val,i,s=85,label=name)
            else: ax.text(317,i,'no warning',va='center',fontsize=9)
        actual=p.First_Actual_High_Cut.dropna()
        if len(actual):ax.axvline(actual.iloc[0],ls='--',color='black',label='First measured crossing')
        ax.set_yticks(range(4),names);ax.set_title(f'{c}: first warnings at 180 µm')
        ax.set_xlim(0,365);ax.grid(axis='x',alpha=.3)
    axs[-1].set_xlabel('Cut number (not replacement simulation)')
    save('F12_first_warning_timing',fig)

# Optional table 12: channel attribution comparison
shap=read('ig_vs_grouped_shap_channels.csv',['Cutter','Cut','Channel','IG_Signed_um','SHAP_Signed_um'])
if shap is not None:
    table(shap,'T12_ig_shap_signed_channels_pilot.csv')
    fig,axs=plt.subplots(1,3,figsize=(13,4))
    for ax,c in zip(axs,['C1','C4','C6']):
        d=shap[(shap.Cutter==c)&(shap.Cut==158)].sort_values('Channel')
        if len(d)!=7:continue
        x=np.arange(7);w=.38
        ax.bar(x-w/2,d.IG_Signed_um,width=w,label='IG')
        ax.bar(x+w/2,d.SHAP_Signed_um,width=w,label='Grouped Shapley')
        ax.set_xticks(x,[str(v) for v in d.Channel]);ax.set_xlabel('Sensor channel');ax.set_ylabel('Signed contribution (µm)');ax.set_title(f'{c}, cut 158')
    axs[0].legend();save('F13_ig_shap_channel_contributions_pilot',fig)

# Figure 14: protocol partition schematic (exact block allocations used in V2)
blocks=np.arange(1,16)
labels=['Calibration' if b in (2,8,14) else 'Validation' if b in (4,12) else 'Training' for b in blocks]
partition_colors={'Training':'#6E93B1','Validation':'#D5A657','Calibration':'#63A58F'}
fig,ax=plt.subplots(figsize=(11,3.2))
for i,(b,label) in enumerate(zip(blocks,labels)):
    ax.barh(0,21,left=21*i,height=.54,color=partition_colors[label],edgecolor='white')
    ax.text(21*i+10.5,0,f'B{b}',va='center',ha='center',color='white',fontsize=9,weight='bold')
ax.set_xlim(0,315);ax.set_yticks([0],['Each training cutter']);ax.set_xlabel('Cut index (15 contiguous blocks × 21 cuts)')
ax.set_xticks([0,63,126,189,252,315]);ax.set_title('Protocol V2 within-cutter training, validation and calibration blocks')
from matplotlib.patches import Patch, FancyBboxPatch, FancyArrowPatch
ax.legend(handles=[Patch(facecolor=partition_colors[l],label=l) for l in ['Training','Validation','Calibration']],loc='upper center',bbox_to_anchor=(.5,-.28),ncol=3,frameon=False)
ax.set_ylim(-.65,.65)
save('F14_protocol_v2_split_blocks',fig)

# Figure 15: research methodology overview (conceptual diagram, not a results claim)
fig,ax=plt.subplots(figsize=(12,3.9));ax.set_xlim(0,12);ax.set_ylim(0,4);ax.axis('off')
nodes=[(.1,1.3,2.0,1.4,'PHM2010\n3 cutters × 315 cuts'),
       (2.55,1.3,2.1,1.4,'Raw sensor windows\n4 × 2,048 × 7'),
       (5.1,1.3,2.0,1.4,'Models\nRF | CNN | Ensemble'),
       (7.55,2.25,2.0,1.0,'Uncertainty\nConformal intervals'),
       (7.55,.55,2.0,1.0,'Explanations\nIG | grouped SHAP'),
       (10.05,1.3,1.8,1.4,'Offline warnings\nThreshold scenarios')]
for x,y,w,h,label in nodes:
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.08',facecolor='#EDF3F7',edgecolor='#48677B',linewidth=1.2))
    ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=9)
arrows=[((2.1,2.0),(2.52,2.0)),((4.65,2.0),(5.07,2.0)),((7.1,2.15),(7.53,2.7)),((7.1,1.85),(7.53,1.05)),((9.55,2.7),(10.03,2.1)),((9.55,1.05),(10.03,1.9))]
for p1,p2 in arrows:
    ax.add_patch(FancyArrowPatch(p1,p2,arrowstyle='-|>',mutation_scale=14,color='#48677B',linewidth=1.2))
ax.set_title('Study design: cross-cutter wear prediction, uncertainty, explainability and decisions',pad=8)
save('F15_study_workflow_schematic',fig)

# Build index and package; all data comes from saved files.
manifest=['PHM2010 PAPER ASSETS — EXPLORATORY RESULTS','',
          'All plots and tables are derived from saved Kaggle experiment CSVs.',
          'PNG files are 300 dpi; editable vector PDF counterparts are included.',
          'No test results were inferred for missing input files.',
          '', 'Scientific caveats:',
          '- Protocol V2 was selected after reviewing preliminary held-out-cutter results.',
          '- Three physical cutters; adjacent cuts are dependent.',
          '- Model ensemble SD is not a calibrated confidence interval.',
          '- Cross-cutter conformal coverage is empirical, not guaranteed.',
          '- Grouped SHAP is a fixed-mean channel-masking game, not a causal explanation.',
          '- Replacement costs are hypothetical and count repeated cut-level warnings.',
          '- First-warning plots are not counterfactual replacement or savings evaluations.',
          '', 'Figures (PNG and PDF):']
manifest += ['  '+v for v in made_figs]
manifest += ['', 'Tables (CSV):']+['  '+v for v in made_tables]
manifest += ['', 'Missing input files:']+(['  '+v for v in sorted(set(missing))] if missing else ['  None'])
(BASE/'README_assets.txt').write_text('\n'.join(manifest),encoding='utf-8')
archive=WORK/'PHM2010_Paper_Assets.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for p in sorted(BASE.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(WORK))
with zipfile.ZipFile(archive) as z:
    bad=z.testzip()
    assert bad is None, f'Archive verification failed: {bad}'
print('\n=== FINISHED ===')
print('PNG figures:',len(made_figs))
print('Tables:',len(made_tables))
print('Assets folder:',BASE)
print('ZIP:',archive)
print('ZIP size (MB):',round(archive.stat().st_size/1024**2,2))
print('Missing inputs:',sorted(set(missing)))


In [ ]:

from pathlib import Path
import zipfile
import json
import datetime
import os

# ==========================================
# COMPLETE RESEARCH BACKUP
# ==========================================

WORK = Path("/kaggle/working")

ZIP_PATH = (
    WORK / "PHM2010_COMPLETE_RESEARCH_BACKUP.zip"
)

# Include research source files,
# results, figures, trained models and data.
ALLOWED_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".csv",
    ".png",
    ".pdf",
    ".pt",
    ".pth",
    ".npz",
    ".txt",
    ".md",
    ".json",
    ".yaml",
    ".yml",
    ".toml"
}

# Do not include ZIPs inside the new ZIP.
# Exclude temporary and unrelated files.
EXCLUDED_DIRECTORIES = {
    "__pycache__",
    ".git",
    ".ipynb_checkpoints"
}

# ==========================================
# 1. SAVE EXECUTED CODE HISTORY
# ==========================================

history_path = WORK / "executed_kaggle_code_history.txt"

try:
    ipython = get_ipython()

    with open(
        history_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(
            "KAGGLE EXECUTED CODE HISTORY\n"
        )
        file.write(
            "Current kernel session only.\n"
        )
        file.write(
            "This is NOT a replacement for "
            "the complete .ipynb notebook.\n\n"
        )

        count = 0

        for session, cell_number, source in (
            ipython.history_manager.get_range(
                output=False,
                raw=True
            )
        ):
            if not source.strip():
                continue

            file.write(
                "\n" + "=" * 65 + "\n"
            )
            file.write(
                f"SESSION {session} | "
                f"CELL {cell_number}\n"
            )
            file.write(
                "=" * 65 + "\n\n"
            )
            file.write(source)
            file.write("\n")

            count += 1

    print(
        "Executed code cells captured:",
        count
    )

except Exception as error:
    print(
        "Could not export executed code history:",
        str(error)
    )

# ==========================================
# 2. COLLECT FILES
# ==========================================

files_to_backup = []

for root, dirs, filenames in os.walk(WORK):

    dirs[:] = [
        d for d in dirs
        if d not in EXCLUDED_DIRECTORIES
    ]

    for filename in filenames:

        path = Path(root) / filename

        if path == ZIP_PATH:
            continue

        # Existing ZIP backups are deliberately
        # excluded to avoid nested archives.
        if path.suffix.lower() == ".zip":
            continue

        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        if not path.is_file():
            continue

        files_to_backup.append(path)

files_to_backup = sorted(set(files_to_backup))

assert files_to_backup, (
    "No files found for backup."
)

# ==========================================
# 3. CREATE MANIFEST
# ==========================================

manifest = {
    "project": "PHM2010 Tool Wear Prediction",
    "backup_created_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "number_of_files": len(files_to_backup),
    "files": [
        {
            "path": str(
                p.relative_to(WORK)
            ),
            "size_bytes": p.stat().st_size
        }
        for p in files_to_backup
    ],
    "notes": [
        "Contains eligible files in /kaggle/working.",
        "Does not include Kaggle /kaggle/input data.",
        "Executed history covers current kernel only.",
        "Download the full .ipynb separately.",
        "Review backup for private information."
    ]
}

manifest_path = (
    WORK / "research_backup_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        manifest,
        file,
        indent=2
    )

files_to_backup.append(manifest_path)

# ==========================================
# 4. CREATE ZIP
# ==========================================

print("\nCREATING COMPLETE RESEARCH BACKUP")

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as archive:

    for path in files_to_backup:

        relative_path = path.relative_to(WORK)

        archive.write(
            path,
            arcname=str(relative_path)
        )

# ==========================================
# 5. VERIFY ZIP
# ==========================================

print("\nVERIFYING ARCHIVE")

with zipfile.ZipFile(
    ZIP_PATH, "r"
) as archive:

    corrupted = archive.testzip()

    assert corrupted is None, (
        f"Corrupted archive entry: {corrupted}"
    )

    archive_names = archive.namelist()

    print(
        "Files inside ZIP:",
        len(archive_names)
    )

# ==========================================
# 6. PRINT BACKUP CONTENTS
# ==========================================

from collections import Counter

counts = Counter(
    p.suffix.lower()
    for p in files_to_backup
)

print("\nFILE TYPES INCLUDED")

for extension, count in sorted(
    counts.items()
):
    print(
        extension,
        ":",
        count
    )

print("\nIMPORTANT FILE CHECK")

required_files = [
    "phm_sequences.npz",
    "phm_protocol_v2_splits.csv",
    "rf_protocol_v2_results.csv",
    "cnn_protocol_v2_results.csv",
    "ensemble_evaluation_results.csv",
    "ensemble_conformal_comparison.csv",
    "explanation_stability_v3.csv",
    "replacement_policy_scenarios.csv",
    "replacement_first_warning_analysis.csv"
]

for filename in required_files:
    found = any(
        p.name == filename
        for p in files_to_backup
    )

    print(
        filename,
        "INCLUDED" if found else "MISSING"
    )

print("\n" + "=" * 55)
print("COMPLETE BACKUP CREATED")
print("=" * 55)

print("ZIP filename:", ZIP_PATH.name)

print(
    "ZIP size:",
    round(
        ZIP_PATH.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

print("Location:", ZIP_PATH)
print("Archive integrity: VERIFIED")

print("\nRemember to download your full")
print("Kaggle .ipynb notebook separately.")
